# Put the SEEA-Sim inputs into the JEM validation folder

This notebook only writes files to Drive; it does not install or run SyncroSim. In **SEEA-Sim JEM validation 2015-2022** it:

1. writes `4_seeasim/`: the SEEA-Sim tables, `add_seeasim.py`, and the package file `stsimseea-1-0-5.ssimpkg` (built from the source below);
2. checks every name in the tables against this folder's ST-Sim tables;
3. adds a few lines to `build_and_test.py` so it loads `4_seeasim/` (the original is kept as `_OLD_build_and_test_before_seeasim.py`). Safe to run again, e.g. after `recalibrate_validation.ipynb` rewrites the folder.

Then on the laptop, in that folder (downloaded or synced):
```
pip install pysyncrosim pandas
python build_and_test.py --full --no-run --ssim "C:\Program Files\SyncroSim Studio"
```
That builds `JEM_cycle1.ssim` with ST-Sim and SEEA-Sim inside; it also installs SEEA-Sim 1.0.5 if SyncroSim doesn't have it yet. Look for `seeasim:` lines in `build_log.txt`, then open the library in Studio and run.

Mount the Google account that owns **Thesis**.

In [1]:
# 1. Mount Drive and find the folder
from google.colab import drive
drive.mount('/content/drive')
import glob, os, re, shutil, zipfile
import pandas as pd
FOLDER_NAME = 'SEEA-Sim JEM validation 2015-2022'
hits = glob.glob(f'/content/drive/MyDrive/**/{FOLDER_NAME}', recursive=True)
print(hits)
assert hits, "Folder not found. If it is in 'Shared with me', add a shortcut to My Drive first, or set FOLDER by hand."
FOLDER = hits[0]
OUT = os.path.join(FOLDER, '4_seeasim')
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive
['/content/drive/MyDrive/work/Thesis/SEEA-Sim JEM validation 2015-2022']


In [2]:
# 2a. SEEA-Sim 1.0.5 source (src/ in github.com/Shlhnj/stsimseea), paths relative to src/
PKG_FILES = {
 "package.xml": "<?xml version=\"1.0\" encoding=\"utf-8\"?>\n<!--\n  SEEA-Sim (package name: stsimseea)\n  SyncroSim add-on that turns ST-Sim 4.5 output (state classes, transitions,\n  stocks and flows) into SEEA Ecosystem Accounting tables.\n\n  Syntax follows ApexRMS/stsimecodep (builtFor, no <dataSheets> wrapper, no\n  explicit ID/ScenarioId columns) and ApexRMS/helloworldPy (Python transformer\n  with condaEnv). ST-Sim datasheet and column names checked against\n  ApexRMS/stsim src/package.xml, version 4.5.4.\n\n  Run it as a second pipeline stage after stsim_Runtime.\n-->\n<package name=\"stsimseea\" displayName=\"SEEA-Sim\" description=\"SEEA Ecosystem Accounting tables from ST-Sim output\" version=\"1.0.5\" minSyncroSimVersion=\"3.1.14\" url=\"https://github.com/Shlhnj/stsimseea\">\n\n  <builtFor package=\"stsim\" version=\"4.5.4\"/>\n\n  <!-- ===================== Project definitions ===================== -->\n\n  <dataSheet name=\"Service\" displayName=\"Ecosystem Service\" dataScope=\"Project\" displayMember=\"Name\">\n    <column name=\"Name\" dataType=\"String\"/>\n    <column name=\"ServiceType\" displayName=\"Type\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Provisioning|2:Regulating|3:Cultural\"/>\n    <column name=\"Units\" displayName=\"Physical Units\" dataType=\"String\" isOptional=\"True\"/>\n    <column name=\"Description\" dataType=\"String\" isOptional=\"True\"/>\n  </dataSheet>\n\n  <!-- ===================== Scenario inputs ===================== -->\n\n  <dataSheet name=\"Options\" displayName=\"Options\" isSingleRow=\"True\">\n    <column name=\"ReportByStratum\" displayName=\"Report by primary stratum (accounting area)\" dataType=\"Boolean\"/>\n    <column name=\"Currency\" dataType=\"String\" isOptional=\"True\"/>\n    <column name=\"MapOutput\" displayName=\"Write value maps (needs ST-Sim state-class maps at the period opening and closing years)\" dataType=\"Boolean\" isOptional=\"True\"/>\n  </dataSheet>\n\n  <dataSheet name=\"AccountingPeriod\" displayName=\"Accounting Periods\">\n    <column name=\"OpeningTimestep\" displayName=\"Opening\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"GreaterEqual\" formula1=\"0\" format=\"d\" allowDbNull=\"False\"/>\n    <column name=\"ClosingTimestep\" displayName=\"Closing\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"GreaterEqual\" formula1=\"0\" format=\"d\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"OpeningTimestep|ClosingTimestep\"/>\n  </dataSheet>\n\n  <dataSheet name=\"TransitionTypeClass\" displayName=\"Transition Type Classification\">\n    <column name=\"TransitionTypeId\" displayName=\"Transition Type\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_TransitionType\" allowDbNull=\"False\"/>\n    <column name=\"ChangeType\" displayName=\"Change Type\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Managed|2:Unmanaged|3:Catastrophic\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"TransitionTypeId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"CarbonStockGroup\" displayName=\"Carbon Stock Groups\">\n    <column name=\"StockGroupId\" displayName=\"Stock Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\" allowDbNull=\"False\"/>\n    <column name=\"Role\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Ecosystem pool|2:Atmosphere|3:Water|4:Long-term burial|5:Other external\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"StockGroupId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"CarbonFlowGroup\" displayName=\"Carbon Flow Groups\">\n    <column name=\"FlowGroupId\" displayName=\"Flow Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_FlowGroup\" allowDbNull=\"False\"/>\n    <validation validationType=\"Unique\" columns=\"FlowGroupId\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ServiceSupply\" displayName=\"Service Supply Rules\">\n    <column name=\"ServiceId\" displayName=\"Service\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\" allowDbNull=\"False\"/>\n    <column name=\"StratumId\" displayName=\"Stratum\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" isOptional=\"True\"/>\n    <column name=\"Basis\" dataType=\"Integer\" validationType=\"List\" formula1=\"1:Area|2:Stock group|3:Flow group\" allowDbNull=\"False\"/>\n    <column name=\"StockGroupId\" displayName=\"Stock Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\" isOptional=\"True\"/>\n    <column name=\"FlowGroupId\" displayName=\"Flow Group\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_FlowGroup\" isOptional=\"True\"/>\n    <column name=\"Coefficient\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ServicePrice\" displayName=\"Service Prices\" hasTimestep=\"True\">\n    <column name=\"ServiceId\" displayName=\"Service\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\" allowDbNull=\"False\"/>\n    <column name=\"Price\" displayName=\"Price per physical unit\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"AssetValuation\" displayName=\"Asset Valuation\">\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" isOptional=\"True\"/>\n    <column name=\"DiscountRate\" displayName=\"Discount Rate\" dataType=\"Double\" validationType=\"Decimal\" validationCondition=\"GreaterEqual\" formula1=\"0.0\" allowDbNull=\"False\"/>\n    <column name=\"AssetLifeYears\" displayName=\"Asset Life (years)\" dataType=\"Integer\" validationType=\"WholeNumber\" validationCondition=\"Greater\" formula1=\"0\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <dataSheet name=\"ConditionChange\" displayName=\"Condition Change\">\n    <column name=\"StateClassId\" displayName=\"State Class\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\" allowDbNull=\"False\"/>\n    <column name=\"OpeningTimestep\" displayName=\"Opening\" dataType=\"Integer\" format=\"d\" isOptional=\"True\"/>\n    <column name=\"ClosingTimestep\" displayName=\"Closing\" dataType=\"Integer\" format=\"d\" isOptional=\"True\"/>\n    <column name=\"Change\" displayName=\"Change in condition index\" dataType=\"Double\" allowDbNull=\"False\"/>\n  </dataSheet>\n\n  <!-- ===================== Outputs ===================== -->\n  <!-- Entry labels and signs follow the UN SEEA EA (2024) tables. Extent\n       (Table 4.1/AI.2) and Carbon Stock Account (Table 13.3): reductions are\n       positive, Net change = additions - reductions. Monetary Asset Account\n       (Table 10.1/AI.11): decreases negative, Net change = sum of entries.\n       \"Group: item\" = item under that group's sub-header in the UN table.\n       Carbon Stock Changes by Process is SEEA-Sim detail; reductions negative. -->\n\n  <dataSheet name=\"OutputExtent\" displayName=\"Extent Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputChangeMatrix\" displayName=\"Ecosystem Type Change Matrix\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"FromStateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ToStateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCarbonAccount\" displayName=\"Carbon Stock Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"StockGroupId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCarbonStock\" displayName=\"Carbon Stock Changes by Process\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"StockGroupId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockGroup\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCarbonFlow\" displayName=\"Carbon Flow Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"TransitionTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_TransitionType\" isOptional=\"True\"/>\n    <column name=\"FromStockTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockType\"/>\n    <column name=\"ToStockTypeId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StockType\"/>\n    <column name=\"Category\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputServiceSupply\" displayName=\"Physical Supply\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ServiceId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputServiceValue\" displayName=\"Monetary Supply\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ServiceId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"Service\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputAssetAccount\" displayName=\"Monetary Asset Account\" hasIteration=\"True\">\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"Entry\" dataType=\"String\"/>\n    <column name=\"Amount\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputAssetValue\" displayName=\"Ecosystem Value by Year\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"StateClassId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_StateClass\"/>\n    <column name=\"ServiceValue\" displayName=\"Ecosystem service value (per year)\" dataType=\"Double\"/>\n    <column name=\"AssetValue\" displayName=\"Ecosystem asset value (NPV)\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputSpatialValue\" displayName=\"Value Maps\" hasIteration=\"True\" hasTimestep=\"True\">\n    <column name=\"ServiceValueFile\" displayName=\"Service value per area unit per year\" dataType=\"String\" isExternalFile=\"True\" externalFileFilter=\"GeoTIFF Files (*.tif)|*.tif|All Files (*.*)|*.*\" isRaster=\"True\"/>\n    <column name=\"AssetValueFile\" displayName=\"Asset value per area unit\" dataType=\"String\" isExternalFile=\"True\" externalFileFilter=\"GeoTIFF Files (*.tif)|*.tif|All Files (*.*)|*.*\" isRaster=\"True\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputWorkbook\" displayName=\"SEEA-Sim Workbook\" isSingleRow=\"True\">\n    <column name=\"File\" displayName=\"Excel workbook\" dataType=\"String\" isExternalFile=\"True\" externalFileFilter=\"Excel Files (*.xlsx)|*.xlsx|All Files (*.*)|*.*\"/>\n  </dataSheet>\n\n  <dataSheet name=\"OutputCheck\" displayName=\"Reconciliation Checks\" hasIteration=\"True\">\n    <column name=\"Account\" dataType=\"String\"/>\n    <column name=\"OpeningTimestep\" dataType=\"Integer\"/>\n    <column name=\"ClosingTimestep\" dataType=\"Integer\"/>\n    <column name=\"StratumId\" dataType=\"Integer\" validationType=\"Datasheet\" formula1=\"stsim_Stratum\" isOptional=\"True\"/>\n    <column name=\"Item\" dataType=\"String\"/>\n    <column name=\"Residual\" dataType=\"Double\"/>\n  </dataSheet>\n\n  <!-- ===================== Transformer ===================== -->\n\n  <transformer\n    name=\"Main\"\n    displayName=\"SEEA-Sim Accounts\"\n    isRunnable=\"True\"\n    programName=\"python\"\n    programArguments=\"transformer.py\"\n    condaEnv=\"stsimseea.yml\"\n    condaEnvVersion=\"3\">\n      <dataSheet name=\"Options\" type=\"Input\"/>\n      <dataSheet name=\"AccountingPeriod\" type=\"Input\"/>\n      <dataSheet name=\"TransitionTypeClass\" type=\"Input\"/>\n      <dataSheet name=\"CarbonStockGroup\" type=\"Input\"/>\n      <dataSheet name=\"CarbonFlowGroup\" type=\"Input\"/>\n      <dataSheet name=\"ServiceSupply\" type=\"Input\"/>\n      <dataSheet name=\"ServicePrice\" type=\"Input\"/>\n      <dataSheet name=\"AssetValuation\" type=\"Input\"/>\n      <dataSheet name=\"ConditionChange\" type=\"Input\"/>\n      <dataSheet name=\"OutputExtent\" type=\"Output\"/>\n      <dataSheet name=\"OutputChangeMatrix\" type=\"Output\"/>\n      <dataSheet name=\"OutputCarbonAccount\" type=\"Output\"/>\n      <dataSheet name=\"OutputCarbonStock\" type=\"Output\"/>\n      <dataSheet name=\"OutputCarbonFlow\" type=\"Output\"/>\n      <dataSheet name=\"OutputServiceSupply\" type=\"Output\"/>\n      <dataSheet name=\"OutputServiceValue\" type=\"Output\"/>\n      <dataSheet name=\"OutputAssetAccount\" type=\"Output\"/>\n      <dataSheet name=\"OutputAssetValue\" type=\"Output\"/>\n      <dataSheet name=\"OutputSpatialValue\" type=\"Output\"/>\n      <dataSheet name=\"OutputWorkbook\" type=\"Output\"/>\n      <dataSheet name=\"OutputCheck\" type=\"Output\"/>\n  </transformer>\n\n  <!-- ===================== Layouts ===================== -->\n\n  <layout type=\"Project\">\n    <item name=\"Service\"/>\n  </layout>\n\n  <layout type=\"Scenario\">\n    <group name=\"SEEASim\" displayName=\"SEEA-Sim\">\n      <item name=\"Options\"/>\n      <item name=\"AccountingPeriod\"/>\n      <item name=\"TransitionTypeClass\"/>\n      <group name=\"Carbon\">\n        <item name=\"CarbonStockGroup\"/>\n        <item name=\"CarbonFlowGroup\"/>\n      </group>\n      <group name=\"Services\" displayName=\"Ecosystem Services\">\n        <item name=\"ServiceSupply\"/>\n        <item name=\"ServicePrice\"/>\n        <item name=\"AssetValuation\"/>\n        <item name=\"ConditionChange\"/>\n      </group>\n      <group name=\"Results\">\n        <item name=\"OutputExtent\"/>\n        <item name=\"OutputChangeMatrix\"/>\n        <item name=\"OutputCarbonAccount\"/>\n        <item name=\"OutputCarbonStock\"/>\n        <item name=\"OutputCarbonFlow\"/>\n        <item name=\"OutputServiceSupply\"/>\n        <item name=\"OutputServiceValue\"/>\n        <item name=\"OutputAssetAccount\"/>\n        <item name=\"OutputAssetValue\"/>\n        <item name=\"OutputSpatialValue\"/>\n        <item name=\"OutputWorkbook\"/>\n        <item name=\"OutputCheck\"/>\n      </group>\n    </group>\n  </layout>\n\n  <layout type=\"Chart\">\n    <group name=\"SEEASim\" displayName=\"SEEA-Sim\">\n      <item name=\"seeasimSupplyByService\" displayName=\"Physical supply (by service)\" dataSheet=\"OutputServiceSupply\" column=\"Amount\" filter=\"ServiceId\" prefixFolderName=\"False\"/>\n      <item name=\"seeasimValueByService\" displayName=\"Monetary supply (by service)\" dataSheet=\"OutputServiceValue\" column=\"Amount\" filter=\"ServiceId\" prefixFolderName=\"False\"/>\n      <item name=\"seeasimValueByClass\" displayName=\"Monetary supply (by ecosystem type)\" dataSheet=\"OutputServiceValue\" column=\"Amount\" filter=\"StateClassId\" prefixFolderName=\"False\"/>\n      <item name=\"seeasimAssetByClass\" displayName=\"Asset value, NPV (by ecosystem type)\" dataSheet=\"OutputAssetValue\" column=\"AssetValue\" filter=\"StateClassId\" prefixFolderName=\"False\"/>\n    </group>\n  </layout>\n\n  <layout type=\"Map\">\n    <group name=\"SEEASim\" displayName=\"SEEA-Sim\">\n      <item name=\"seeasimValueMap\" displayName=\"Service value per area unit per year\" dataSheet=\"OutputSpatialValue\" column=\"ServiceValueFile\"/>\n      <item name=\"seeasimAssetMap\" displayName=\"Asset value per area unit\" dataSheet=\"OutputSpatialValue\" column=\"AssetValueFile\"/>\n    </group>\n  </layout>\n\n</package>\n",
 "seea_accounts.py": "\"\"\"\nSEEA-Sim accounting core.\n\nPure pandas. No SyncroSim import, so it can be tested anywhere. The\ntransformer (transformer.py) reads ST-Sim datasheets, passes them here as\nDataFrames and writes the results back.\n\nInputs use ST-Sim 4.5 output column names:\n  stsim_OutputStratumState          Iteration, Timestep, StratumId, StateClassId, Amount\n  stsim_OutputStratumTransitionState Iteration, Timestep, StratumId, TransitionTypeId,\n                                    StateClassId, EndStateClassId, Amount\n  stsim_OutputStock                 Iteration, Timestep, StratumId, StateClassId,\n                                    StockGroupId, Amount\n  stsim_OutputFlow                  Iteration, Timestep, FromStratumId, FromStateClassId,\n                                    FromStockTypeId, ToStockTypeId, TransitionTypeId,\n                                    FlowGroupId, Amount\nId columns can hold names or integers; values are only used as keys.\n\nConventions\n  * A period (o, c) covers the transitions and flows of timesteps o+1 .. c.\n    Opening = state at the end of timestep o, closing = state at the end of c.\n  * Entry labels and signs follow the UN SEEA EA (2024) tables. Physical\n    accounts (extent, Table 4.1/AI.2; carbon stock, Table 13.3) record\n    reductions as positive amounts: Net change = additions - reductions.\n    The monetary asset account (Table 10.1/AI.11) records decreases as\n    negative amounts: Net change = sum of the change entries. The carbon\n    process detail (carbon_accounts' first table) is not a UN table and\n    uses negative reductions.\n  * Every account is computed per iteration. SyncroSim summarises across\n    iterations.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nALL = \"__all__\"  # placeholder stratum when results are not split by stratum\n\nMANAGED, UNMANAGED, CATASTROPHIC = \"Managed\", \"Unmanaged\", \"Catastrophic\"\nCHANGE_TYPES = {1: MANAGED, 2: UNMANAGED, 3: CATASTROPHIC}\n\nPOOL, ATMOSPHERE, WATER, BURIAL, OTHER = (\n    \"Ecosystem pool\", \"Atmosphere\", \"Water\", \"Long-term burial\", \"Other external\")\nROLES = {1: POOL, 2: ATMOSPHERE, 3: WATER, 4: BURIAL, 5: OTHER}\n\nAREA, STOCK, FLOW = \"Area\", \"Stock group\", \"Flow group\"\nBASES = {1: AREA, 2: STOCK, 3: FLOW}\n\nPERIOD = [\"OpeningTimestep\", \"ClosingTimestep\"]\n\n\nclass AccountError(ValueError):\n    \"\"\"Input problem the user has to fix; the message says how.\"\"\"\n\n\ndef _log(log, msg):\n    (log or print)(msg)\n\n\ndef list_value(v, mapping: dict, what: str):\n    \"\"\"SyncroSim List columns may arrive as 1, '1' or the label.\"\"\"\n    if pd.isna(v):\n        raise AccountError(f\"{what} is empty.\")\n    labels = {s.lower(): s for s in mapping.values()}\n    if isinstance(v, str) and v.strip().lower() in labels:\n        return labels[v.strip().lower()]\n    try:\n        return mapping[int(float(v))]\n    except (ValueError, KeyError):\n        raise AccountError(f\"Unknown {what}: {v!r}. Allowed: {list(mapping.values())}\")\n\n\ndef annuity_factor(rate: float, life: int) -> float:\n    \"\"\"Present value of 1 per year for `life` years at `rate`.\"\"\"\n    if rate == 0:\n        return float(life)\n    return (1 - (1 + rate) ** -life) / rate\n\n\ndef _need(df: pd.DataFrame, cols, name: str):\n    missing = [c for c in cols if c not in df.columns]\n    if missing:\n        raise AccountError(f\"{name} is missing column(s) {missing}; found {list(df.columns)}.\")\n\n\ndef _stratum(df: pd.DataFrame, col: str, by_stratum: bool) -> pd.DataFrame:\n    df = df.copy()\n    df[\"StratumId\"] = df[col] if by_stratum else ALL\n    return df\n\n\ndef _finish(df: pd.DataFrame) -> pd.DataFrame:\n    \"\"\"Replace the placeholder stratum with null before saving.\"\"\"\n    if \"StratumId\" in df.columns:\n        df[\"StratumId\"] = df[\"StratumId\"].astype(object).where(df[\"StratumId\"] != ALL, None)\n    return df.reset_index(drop=True)\n\n\ndef default_periods(state: pd.DataFrame) -> list[tuple[int, int]]:\n    return [(int(state[\"Timestep\"].min()), int(state[\"Timestep\"].max()))]\n\n\ndef _check_timesteps(df: pd.DataFrame, timesteps, name: str):\n    have = set(df[\"Timestep\"].unique())\n    missing = sorted(t for t in timesteps if t not in have)\n    if missing:\n        raise AccountError(\n            f\"{name} has no output for timestep(s) {missing}. Set its summary output \"\n            f\"frequency so these timesteps are written (every timestep is safest).\")\n\n\n# ---------------------------------------------------------------- extent\n\n# SEEA EA Table 4.1 / AI.2. \"Group: item\" = item under that group's sub-header.\nEXTENT_ENTRIES = [\n    \"Opening extent\",\n    \"Additions to extent: Managed expansions\",\n    \"Additions to extent: Unmanaged expansions\",\n    \"Reductions in extent: Managed reductions\",\n    \"Reductions in extent: Unmanaged reductions\",\n    \"Net change in extent\",\n    \"Closing extent\",\n]\nEXTENT_REDUCTIONS = [e for e in EXTENT_ENTRIES if e.startswith(\"Reductions in extent\")]\n\n\ndef _prep_state(state, by_stratum):\n    _need(state, [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"Amount\"],\n          \"stsim_OutputStratumState\")\n    df = _stratum(state, \"StratumId\", by_stratum)\n    return df.groupby([\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\"],\n                      as_index=False)[\"Amount\"].sum()\n\n\ndef _prep_trans(trans, tt_class: dict, by_stratum, log):\n    _need(trans, [\"Iteration\", \"Timestep\", \"StratumId\", \"TransitionTypeId\",\n                  \"StateClassId\", \"EndStateClassId\", \"Amount\"],\n          \"stsim_OutputStratumTransitionState\")\n    df = _stratum(trans, \"StratumId\", by_stratum)\n    df = df[df[\"StateClassId\"] != df[\"EndStateClassId\"]].copy()\n    unlisted = sorted({str(t) for t in df[\"TransitionTypeId\"].unique() if t not in tt_class})\n    if unlisted:\n        _log(log, \"SEEA-Sim: transition types not in Transition Type Classification are \"\n                  f\"treated as Unmanaged: {', '.join(unlisted)}\")\n    df[\"ChangeType\"] = df[\"TransitionTypeId\"].map(tt_class).fillna(UNMANAGED)\n    return df\n\n\ndef extent_components(state, trans, periods, tt_class, by_stratum=False, log=None):\n    \"\"\"\n    One row per (Iteration, StratumId, period, StateClassId) with\n    Opening, Closing, Add_<type>, Red_<type> (positive areas) and Residual\n    (= closing - opening - additions + reductions; zero unless something\n    other than recorded transitions changed the class areas).\n    \"\"\"\n    st = _prep_state(state, by_stratum)\n    tr = _prep_trans(trans, tt_class, by_stratum, log)\n    keys = [\"Iteration\", \"StratumId\"]\n    out = []\n    for o, c in periods:\n        _check_timesteps(st, [o, c], \"stsim_OutputStratumState\")\n        op = st[st.Timestep == o].drop(columns=\"Timestep\").rename(columns={\"Amount\": \"Opening\"})\n        cl = st[st.Timestep == c].drop(columns=\"Timestep\").rename(columns={\"Amount\": \"Closing\"})\n        t = tr[(tr.Timestep > o) & (tr.Timestep <= c)]\n        add = (t.groupby(keys + [\"EndStateClassId\", \"ChangeType\"])[\"Amount\"].sum()\n                .unstack(\"ChangeType\").add_prefix(\"Add_\").reset_index()\n                .rename(columns={\"EndStateClassId\": \"StateClassId\"}))\n        red = (t.groupby(keys + [\"StateClassId\", \"ChangeType\"])[\"Amount\"].sum()\n                .unstack(\"ChangeType\").add_prefix(\"Red_\").reset_index())\n        df = op.merge(cl, on=keys + [\"StateClassId\"], how=\"outer\")\n        for part in (add, red):\n            if not part.empty:\n                df = df.merge(part, on=keys + [\"StateClassId\"], how=\"outer\")\n        for p in (\"Add_\", \"Red_\"):\n            for ct in (MANAGED, UNMANAGED, CATASTROPHIC):\n                if p + ct not in df.columns:\n                    df[p + ct] = 0.0\n        df = df.fillna({c_: 0.0 for c_ in df.columns if c_ not in keys + [\"StateClassId\"]})\n        df[\"OpeningTimestep\"], df[\"ClosingTimestep\"] = o, c\n        out.append(df)\n    comp = pd.concat(out, ignore_index=True)\n    adds = comp[[f\"Add_{t}\" for t in (MANAGED, UNMANAGED, CATASTROPHIC)]].sum(axis=1)\n    reds = comp[[f\"Red_{t}\" for t in (MANAGED, UNMANAGED, CATASTROPHIC)]].sum(axis=1)\n    comp[\"Residual\"] = comp[\"Closing\"] - comp[\"Opening\"] - adds + reds\n    return comp\n\n\ndef extent_account(comp: pd.DataFrame) -> pd.DataFrame:\n    vals = {\n        \"Opening extent\": comp[\"Opening\"],\n        \"Additions to extent: Managed expansions\": comp[\"Add_Managed\"],\n        \"Additions to extent: Unmanaged expansions\": comp[\"Add_Unmanaged\"] + comp[\"Add_Catastrophic\"],\n        \"Reductions in extent: Managed reductions\": comp[\"Red_Managed\"],\n        \"Reductions in extent: Unmanaged reductions\": comp[\"Red_Unmanaged\"] + comp[\"Red_Catastrophic\"],\n        \"Net change in extent\": comp[\"Closing\"] - comp[\"Opening\"],\n        \"Closing extent\": comp[\"Closing\"],\n    }\n    base = comp[[\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\"]]\n    rows = [base.assign(Entry=e, Amount=vals[e].values) for e in EXTENT_ENTRIES]\n    return _finish(_order(pd.concat(rows, ignore_index=True), EXTENT_ENTRIES))\n\n\ndef _order(df, entries, keys=(\"StateClassId\",)):\n    \"\"\"Sort rows by iteration, stratum, period, keys, then the UN entry order.\"\"\"\n    df = df.copy()\n    df[\"_o\"] = df[\"Entry\"].map({e: i for i, e in enumerate(entries)})\n    by = [c for c in [\"Iteration\", \"StratumId\", *PERIOD, *keys] if c in df.columns]\n    df[\"_s\"] = df[\"StratumId\"].astype(str) if \"StratumId\" in df.columns else \"\"\n    by = [\"_s\" if c == \"StratumId\" else c for c in by]\n    df = df.sort_values(by + [\"_o\"], kind=\"stable\").drop(columns=[\"_o\", \"_s\"])\n    return df.reset_index(drop=True)\n\n\ndef change_matrix(state, trans, periods, tt_class, by_stratum=False, log=None):\n    \"\"\"\n    From x To areas per period. Off-diagonal cells are gross transitions\n    summed over the period's timesteps; the diagonal is opening area minus\n    everything that left the class. Row totals equal opening extent and\n    column totals equal closing extent. A cell that changes more than once\n    in a period (A to B to A) is counted in every move, so off-diagonals are\n    gross flows, not a map-to-map comparison, and the diagonal understates\n    the area that never changed.\n    \"\"\"\n    st = _prep_state(state, by_stratum)\n    tr = _prep_trans(trans, tt_class, by_stratum, log=lambda m: None)\n    keys = [\"Iteration\", \"StratumId\"]\n    out, checks = [], []\n    for o, c in periods:\n        t = tr[(tr.Timestep > o) & (tr.Timestep <= c)]\n        off = (t.groupby(keys + [\"StateClassId\", \"EndStateClassId\"], as_index=False)[\"Amount\"].sum()\n                .rename(columns={\"StateClassId\": \"FromStateClassId\",\n                                 \"EndStateClassId\": \"ToStateClassId\"}))\n        op = st[st.Timestep == o].drop(columns=\"Timestep\")\n        left = off.groupby(keys + [\"FromStateClassId\"], as_index=False)[\"Amount\"].sum()\n        left = left.rename(columns={\"FromStateClassId\": \"StateClassId\", \"Amount\": \"Left\"})\n        diag = op.merge(left, on=keys + [\"StateClassId\"], how=\"left\").fillna({\"Left\": 0.0})\n        diag = diag.assign(FromStateClassId=diag.StateClassId, ToStateClassId=diag.StateClassId,\n                           Amount=diag.Amount - diag.Left)\n        m = pd.concat([diag[keys + [\"FromStateClassId\", \"ToStateClassId\", \"Amount\"]], off],\n                      ignore_index=True)\n        m[\"OpeningTimestep\"], m[\"ClosingTimestep\"] = o, c\n        out.append(m)\n        col = m.groupby(keys + [\"ToStateClassId\"], as_index=False)[\"Amount\"].sum()\n        cl = st[st.Timestep == c].drop(columns=\"Timestep\").rename(\n            columns={\"StateClassId\": \"ToStateClassId\", \"Amount\": \"Closing\"})\n        chk = col.merge(cl, on=keys + [\"ToStateClassId\"], how=\"outer\").fillna(0.0)\n        chk[\"Residual\"] = (chk.Amount - chk.Closing).abs()\n        chk = chk.groupby(keys, as_index=False)[\"Residual\"].sum()\n        chk = chk.assign(Account=\"Change matrix\", OpeningTimestep=o, ClosingTimestep=c,\n                         Item=\"Column totals minus closing extent (should be 0)\")\n        checks.append(chk)\n    return _finish(pd.concat(out, ignore_index=True)), pd.concat(checks, ignore_index=True)\n\n\n# ---------------------------------------------------------------- carbon\n\n# Carbon by process (SEEA-Sim detail, not a UN table; reductions negative).\nSTOCK_ENTRIES = [\n    \"Opening stock\",\n    \"Additions: sequestration\",\n    \"Additions: other inputs\",\n    \"Transfers in from other pools\",\n    \"Reductions: emissions (respiration and decomposition)\",\n    \"Reductions: emissions (conversion and disturbance)\",\n    \"Reductions: export to water\",\n    \"Reductions: transfer to long-term burial\",\n    \"Reductions: other losses\",\n    \"Transfers out to other pools\",\n    \"Change from ecosystem conversion\",\n    \"Net change in stock\",\n    \"Closing stock\",\n]\n\n\n# SEEA EA Table 13.3 (carbon stock account) rows; reductions positive.\nCARBON_ENTRIES = [\n    \"Opening stock\",\n    \"Additions to stock: Unmanaged expansion\",\n    \"Additions to stock: Managed expansion\",\n    \"Additions to stock: Discoveries\",\n    \"Additions to stock: Reclassifications\",\n    \"Additions to stock: Imports\",\n    \"Reductions in stock: Unmanaged contraction\",\n    \"Reductions in stock: Managed contraction\",\n    \"Reductions in stock: Reclassifications\",\n    \"Reductions in stock: Exports\",\n    \"Reductions in stock: Catastrophic losses\",\n    \"Net carbon balance\",\n    \"Closing stock\",\n]\nCARBON_REDUCTIONS = [e for e in CARBON_ENTRIES if e.startswith(\"Reductions in stock\")]\n\n\ndef stock_type_roles(groups: pd.DataFrame, membership: pd.DataFrame):\n    \"\"\"\n    groups: StockGroupId, Role. membership: StockTypeId, StockGroupId\n    (stsim_StockTypeGroupMembership). Returns ({type: role}, {pool type: pool group}).\n    An auto-generated group named '<type> [Type]' with no membership rows is\n    read as containing that one type.\n    \"\"\"\n    role_of, pool_of = {}, {}\n    for _, g in groups.iterrows():\n        gid, role = g[\"StockGroupId\"], list_value(g[\"Role\"], ROLES, \"carbon stock group role\")\n        types = list(membership.loc[membership[\"StockGroupId\"] == gid, \"StockTypeId\"])\n        if not types and isinstance(gid, str) and gid.endswith(\" [Type]\"):\n            types = [gid[: -len(\" [Type]\")]]\n        if not types:\n            raise AccountError(\n                f\"Carbon stock group '{gid}' has no stock types in Stock Type-Group \"\n                f\"Membership, so its flows cannot be classified.\")\n        for t in types:\n            if t in role_of and (role_of[t] != role or role == POOL):\n                raise AccountError(\n                    f\"Stock type '{t}' is in more than one listed carbon stock group. \"\n                    f\"List groups that do not overlap (e.g. AGB, BGB, Soil, not also Total).\")\n            role_of[t] = role\n            if role == POOL:\n                pool_of[t] = gid\n    return role_of, pool_of\n\n\ndef select_flow_rows(flow: pd.DataFrame, flow_groups: list, flow_membership=None):\n    \"\"\"\n    OutputFlow has one row per flow group, so a flow type in two groups is\n    reported twice. Use only the listed groups (they must not share flow\n    types) or, if none are listed, ST-Sim's auto groups '<type> [Type]'.\n    \"\"\"\n    if flow_groups:\n        if flow_membership is not None and not flow_membership.empty:\n            fm = flow_membership[flow_membership[\"FlowGroupId\"].isin(flow_groups)]\n            dup = fm[fm.duplicated(\"FlowTypeId\", keep=False)]\n            if not dup.empty:\n                raise AccountError(\n                    \"These flow types are in more than one listed carbon flow group and \"\n                    f\"would be counted twice: {sorted(map(str, dup.FlowTypeId.unique()))}\")\n        rows = flow[flow[\"FlowGroupId\"].isin(flow_groups)]\n    else:\n        auto = flow[\"FlowGroupId\"].astype(str).str.endswith(\" [Type]\")\n        if not auto.any():\n            raise AccountError(\n                \"No Carbon Flow Groups listed and no auto flow groups ('<type> [Type]') in \"\n                \"stsim_OutputFlow. List flow groups that together hold every carbon flow once.\")\n        rows = flow[auto]\n    return rows\n\n\ndef classify_flows(flow, role_of, by_stratum):\n    _need(flow, [\"Iteration\", \"Timestep\", \"FromStratumId\", \"FromStateClassId\", \"FromStockTypeId\",\n                 \"ToStockTypeId\", \"TransitionTypeId\", \"Amount\"], \"stsim_OutputFlow\")\n    if flow[\"FromStockTypeId\"].isna().all() or flow[\"ToStockTypeId\"].isna().all():\n        raise AccountError(\n            \"stsim_OutputFlow has no From/To stock types. In Stock-Flow Output options, \"\n            \"untick 'omit From stock type' and 'omit To stock type'.\")\n    df = _stratum(flow, \"FromStratumId\", by_stratum).rename(columns={\"FromStateClassId\": \"StateClassId\"})\n    df[\"FromRole\"] = df[\"FromStockTypeId\"].map(role_of).fillna(OTHER)\n    df[\"ToRole\"] = df[\"ToStockTypeId\"].map(role_of).fillna(OTHER)\n    fp, tp = df.FromRole == POOL, df.ToRole == POOL\n    by_transition = df[\"TransitionTypeId\"].notna()\n    cat = np.select(\n        [fp & tp,\n         ~fp & tp & (df.FromRole == ATMOSPHERE),\n         ~fp & tp,\n         fp & (df.ToRole == ATMOSPHERE) & by_transition,\n         fp & (df.ToRole == ATMOSPHERE),\n         fp & (df.ToRole == WATER),\n         fp & (df.ToRole == BURIAL),\n         fp],\n        [\"Internal transfer\",\n         \"Sequestration\",\n         \"Other inputs\",\n         \"Emissions (conversion and disturbance)\",\n         \"Emissions (respiration and decomposition)\",\n         \"Export to water\",\n         \"Transfer to long-term burial\",\n         \"Other losses\"],\n        default=\"\")\n    df[\"Category\"] = cat\n    return df[df.Category != \"\"]\n\n\ndef carbon_accounts(stock, flow, periods, groups, membership, flow_groups=None,\n                    flow_membership=None, by_stratum=False, log=None, tt_class=None,\n                    with_un=False):\n    \"\"\"\n    Returns (stock changes by process, flow account, checks), or with\n    with_un=True also the SEEA EA Table 13.3 account as a fourth item\n    (None for all if no carbon stock groups are listed).\n\n    Table 13.3 entries, per class and ecosystem pool:\n      flows into the pool       -> Managed expansion if the flow was triggered by a\n                                   Managed transition type, else Unmanaged expansion\n      flows out of the pool     -> Managed contraction (Managed transition),\n                                   Catastrophic losses (Catastrophic transition),\n                                   else Unmanaged contraction\n      carbon that moved with area converted to or from the class\n                                -> Reclassifications (net: additions if it gained,\n                                   reductions if it lost); zero over all classes\n      Discoveries, Imports, Exports are geocarbon or economy entries: always 0.\n    Transfers between pools of the same class appear in each pool and cancel\n    in the sum over pools.\n    \"\"\"\n    tt_class = tt_class or {}\n    if groups is None or groups.empty:\n        _log(log, \"SEEA-Sim: no Carbon Stock Groups listed; carbon accounts skipped.\")\n        return (None, None, None, None) if with_un else (None, None, None)\n    role_of, pool_of = stock_type_roles(groups, membership)\n    pools = set(pool_of.values())\n    if not pools:\n        raise AccountError(\"No carbon stock group has Role = Ecosystem pool.\")\n    _need(stock, [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"StockGroupId\", \"Amount\"],\n          \"stsim_OutputStock\")\n    st = _stratum(stock[stock[\"StockGroupId\"].isin(pools)], \"StratumId\", by_stratum)\n    st = st.groupby([\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\", \"StockGroupId\"],\n                    as_index=False)[\"Amount\"].sum()\n    fl = classify_flows(select_flow_rows(flow, flow_groups, flow_membership), role_of, by_stratum)\n\n    fl[\"Mode\"] = [tt_class.get(t, UNMANAGED) if pd.notna(t) else UNMANAGED\n                  for t in fl[\"TransitionTypeId\"]]\n\n    keys = [\"Iteration\", \"StratumId\", \"StateClassId\", \"StockGroupId\"]\n    stock_rows, flow_rows, un_rows, checks = [], [], [], []\n    for o, c in periods:\n        _check_timesteps(st, [o, c], \"stsim_OutputStock\")\n        f = fl[(fl.Timestep > o) & (fl.Timestep <= c)]\n\n        fa = (f.groupby([\"Iteration\", \"StratumId\", \"StateClassId\", \"TransitionTypeId\",\n                         \"FromStockTypeId\", \"ToStockTypeId\", \"Category\"],\n                        as_index=False, dropna=False)[\"Amount\"].sum())\n        fa[\"OpeningTimestep\"], fa[\"ClosingTimestep\"] = o, c\n        flow_rows.append(fa)\n\n        inflow = f[f.ToRole == POOL].assign(StockGroupId=lambda d: d.ToStockTypeId.map(pool_of))\n        inflow[\"Entry\"] = inflow.Category.map({\n            \"Sequestration\": \"Additions: sequestration\",\n            \"Other inputs\": \"Additions: other inputs\",\n            \"Internal transfer\": \"Transfers in from other pools\"})\n        outflow = f[f.FromRole == POOL].assign(StockGroupId=lambda d: d.FromStockTypeId.map(pool_of))\n        outflow[\"Entry\"] = outflow.Category.map(\n            lambda x: \"Transfers out to other pools\" if x == \"Internal transfer\"\n            else f\"Reductions: {x[0].lower()}{x[1:]}\")\n        outflow = outflow.assign(Amount=-outflow.Amount)\n        moves = (pd.concat([inflow, outflow])\n                 .groupby(keys + [\"Entry\"])[\"Amount\"].sum().unstack(\"Entry\"))\n\n        op = st[st.Timestep == o].set_index(keys)[\"Amount\"].rename(\"Opening stock\")\n        cl = st[st.Timestep == c].set_index(keys)[\"Amount\"].rename(\"Closing stock\")\n        w = pd.concat([op, cl, moves], axis=1).fillna(0.0)\n        for e in STOCK_ENTRIES:\n            if e not in w.columns:\n                w[e] = 0.0\n        change_cols = [e for e in STOCK_ENTRIES[1:10]]\n        w[\"Net change in stock\"] = w[\"Closing stock\"] - w[\"Opening stock\"]\n        w[\"Change from ecosystem conversion\"] = w[\"Net change in stock\"] - w[change_cols].sum(axis=1)\n        w = w[STOCK_ENTRIES]\n\n        # SEEA EA Table 13.3\n        inn = inflow.assign(Entry=np.where(inflow.Mode == MANAGED,\n                                           \"Additions to stock: Managed expansion\",\n                                           \"Additions to stock: Unmanaged expansion\"))\n        out_ = outflow.assign(Amount=-outflow.Amount, Entry=np.select(\n            [outflow.Mode == MANAGED, outflow.Mode == CATASTROPHIC],\n            [\"Reductions in stock: Managed contraction\", \"Reductions in stock: Catastrophic losses\"],\n            default=\"Reductions in stock: Unmanaged contraction\"))\n        u = (pd.concat([inn, out_]).groupby(keys + [\"Entry\"])[\"Amount\"].sum().unstack(\"Entry\")\n             .reindex(w.index).fillna(0.0))\n        for e in CARBON_ENTRIES:\n            if e not in u.columns:\n                u[e] = 0.0\n        conv = w[\"Change from ecosystem conversion\"]\n        u[\"Additions to stock: Reclassifications\"] = conv.clip(lower=0) + 0.0\n        u[\"Reductions in stock: Reclassifications\"] = (-conv).clip(lower=0) + 0.0\n        u[\"Opening stock\"], u[\"Closing stock\"] = w[\"Opening stock\"], w[\"Closing stock\"]\n        u[\"Net carbon balance\"] = w[\"Net change in stock\"]\n        u = u[CARBON_ENTRIES]\n        u.columns.name = \"Entry\"\n        ul = u.stack().rename(\"Amount\").reset_index()\n        ul[\"OpeningTimestep\"], ul[\"ClosingTimestep\"] = o, c\n        un_rows.append(ul)\n\n        w.columns.name = \"Entry\"\n        long = w.stack().rename(\"Amount\").reset_index()\n        long[\"OpeningTimestep\"], long[\"ClosingTimestep\"] = o, c\n        stock_rows.append(long)\n\n        # Over all classes the conversion term must cancel.\n        tot = (w[\"Change from ecosystem conversion\"].groupby([\"Iteration\", \"StratumId\"]).sum()\n               .rename(\"Residual\").reset_index())\n        checks.append(tot.assign(Account=\"Carbon stock\", OpeningTimestep=o, ClosingTimestep=c,\n                                 Item=\"Change from ecosystem conversion summed over classes \"\n                                      \"and pools (should be 0)\"))\n    stock_acc = pd.concat(stock_rows, ignore_index=True)\n    stock_acc[\"Entry\"] = pd.Categorical(stock_acc[\"Entry\"], STOCK_ENTRIES, ordered=True)\n    stock_acc = stock_acc.sort_values(keys[:2] + PERIOD + keys[2:] + [\"Entry\"])\n    stock_acc[\"Entry\"] = stock_acc[\"Entry\"].astype(str)\n    result = (_finish(stock_acc), _finish(pd.concat(flow_rows, ignore_index=True)),\n              pd.concat(checks, ignore_index=True))\n    if not with_un:\n        return result\n    un = _order(pd.concat(un_rows, ignore_index=True), CARBON_ENTRIES,\n                keys=(\"StateClassId\", \"StockGroupId\"))\n    return (*result, _finish(un))\n\n\n# ---------------------------------------------------------------- services\n\ndef service_supply(state, stock, flow, rules, by_stratum=False, log=None):\n    \"\"\"\n    Physical supply per (Iteration, Timestep, StratumId, StateClassId, ServiceId).\n    Basis Area:        Coefficient x class area           (units per area unit per year)\n    Basis Stock group: Coefficient x stock of the group   (e.g. retention)\n    Basis Flow group:  Coefficient x flows of the group, attributed to FromStateClassId\n    Rules can be limited to one stratum and/or one state class.\n    The run's first timestep is the initial condition and has no flows, so\n    supply starts at the second timestep for every basis.\n    \"\"\"\n    if rules is None or rules.empty:\n        return None\n    first = state[\"Timestep\"].min()\n    parts = []\n    for _, r in rules.iterrows():\n        basis = list_value(r[\"Basis\"], BASES, \"service supply basis\")\n        if basis == AREA:\n            src = state\n            scol, ccol = \"StratumId\", \"StateClassId\"\n        elif basis == STOCK:\n            if pd.isna(r.get(\"StockGroupId\")):\n                raise AccountError(f\"Service '{r.ServiceId}': basis Stock group needs a Stock Group.\")\n            if stock is None:\n                raise AccountError(\"Stock-based service but stsim_OutputStock is empty.\")\n            src = stock[stock.StockGroupId == r.StockGroupId]\n            scol, ccol = \"StratumId\", \"StateClassId\"\n        else:\n            if pd.isna(r.get(\"FlowGroupId\")):\n                raise AccountError(f\"Service '{r.ServiceId}': basis Flow group needs a Flow Group.\")\n            if flow is None:\n                raise AccountError(\"Flow-based service but stsim_OutputFlow is empty.\")\n            src = flow[flow.FlowGroupId == r.FlowGroupId]\n            scol, ccol = \"FromStratumId\", \"FromStateClassId\"\n        if pd.notna(r.get(\"StratumId\")):\n            src = src[src[scol] == r.StratumId]\n        if pd.notna(r.get(\"StateClassId\")):\n            src = src[src[ccol] == r.StateClassId]\n        src = src[src.Timestep > first]\n        if src.empty:\n            continue\n        d = _stratum(src, scol, by_stratum).rename(columns={ccol: \"StateClassId\"})\n        d = d.groupby([\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\"], as_index=False)[\"Amount\"].sum()\n        d[\"Amount\"] *= float(r[\"Coefficient\"])\n        d[\"ServiceId\"] = r[\"ServiceId\"]\n        parts.append(d)\n    if not parts:\n        _log(log, \"SEEA-Sim: service supply rules matched no output rows.\")\n        return None\n    out = (pd.concat(parts).groupby([\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"ServiceId\"],\n                                    as_index=False)[\"Amount\"].sum())\n    return out\n\n\ndef price_table(prices: pd.DataFrame, services, timesteps, log=None) -> pd.DataFrame:\n    \"\"\"\n    ServiceId x Timestep -> Price. A price row applies from its Timestep\n    onward until the next one; a row with no Timestep applies before the\n    first dated row (or always, if none is dated).\n    \"\"\"\n    rows = []\n    for s in services:\n        p = prices[prices.ServiceId == s] if prices is not None and not prices.empty else prices\n        if p is None or p.empty:\n            _log(log, f\"SEEA-Sim: no price for service '{s}'; left out of monetary tables.\")\n            continue\n        dated = p[p[\"Timestep\"].notna()].sort_values(\"Timestep\") if \"Timestep\" in p else p.iloc[0:0]\n        undated = p[p[\"Timestep\"].isna()] if \"Timestep\" in p else p\n        for t in timesteps:\n            hit = dated[dated.Timestep <= t]\n            if not hit.empty:\n                price = hit.iloc[-1].Price\n            elif not undated.empty:\n                price = undated.iloc[0].Price\n            else:\n                price = dated.iloc[0].Price  # before the first dated price: use it\n            rows.append({\"ServiceId\": s, \"Timestep\": t, \"Price\": float(price)})\n    return pd.DataFrame(rows, columns=[\"ServiceId\", \"Timestep\", \"Price\"])\n\n\ndef service_value(supply, prices_tbl):\n    if supply is None or prices_tbl is None or prices_tbl.empty:\n        return None\n    v = supply.merge(prices_tbl, on=[\"ServiceId\", \"Timestep\"], how=\"inner\")\n    v[\"Amount\"] = v[\"Amount\"] * v[\"Price\"]\n    return v.drop(columns=\"Price\")\n\n\n# ---------------------------------------------------------------- assets\n\n# SEEA EA Table 10.1 / AI.11; decreases negative.\nOCV = \"Other changes in volume of ecosystem assets\"\nCONVERSION_ADD = \"Ecosystem conversions: Additions\"\nCONVERSION_RED = \"Ecosystem conversions: Reductions\"\nCATASTROPHIC_LOSS = f\"{OCV}: Catastrophic losses\"\nUPWARD = f\"{OCV}: Upward reappraisals\"\nDOWNWARD = f\"{OCV}: Downward reappraisals\"\nOTHER_VOLUME = f\"{OCV}: Other changes n.e.c.\"\nASSET_ENTRIES = [\n    \"Opening value\",\n    \"Ecosystem enhancement\",\n    \"Ecosystem degradation\",\n    CONVERSION_ADD,\n    CONVERSION_RED,\n    CATASTROPHIC_LOSS,\n    UPWARD,\n    DOWNWARD,\n    OTHER_VOLUME,\n    \"Revaluations\",\n    \"Net change in value\",\n    \"Closing value\",\n]\n\n\ndef condition_lookup(condition: pd.DataFrame | None):\n    \"\"\"\n    condition: StateClassId, optional OpeningTimestep/ClosingTimestep, Change.\n    Returns f(class, o, c) -> change or None. A row for the exact period wins\n    over a row with no period.\n    \"\"\"\n    if condition is None or condition.empty:\n        return lambda cls, o, c: None\n    d = condition.copy()\n    for col in (\"OpeningTimestep\", \"ClosingTimestep\"):\n        if col not in d.columns:\n            d[col] = np.nan\n    exact, anyp = {}, {}\n    for r in d.itertuples():\n        if pd.notna(r.OpeningTimestep) and pd.notna(r.ClosingTimestep):\n            exact[(r.StateClassId, int(r.OpeningTimestep), int(r.ClosingTimestep))] = float(r.Change)\n        else:\n            anyp[r.StateClassId] = float(r.Change)\n    return lambda cls, o, c: exact.get((cls, o, c), anyp.get(cls))\n\n\ndef classify_volume(volume: float, cond):\n    \"\"\"\n    SEEA EA Annex 10.1 (as tabulated in the SEEALand example):\n      volume > 0 and condition > 0          -> enhancement\n      volume > 0 and condition <= 0         -> upward reappraisal\n      volume < 0 and condition < 0          -> degradation\n      volume < 0 and condition >= 0         -> downward reappraisal\n    With no condition data, a gain is enhancement and a loss degradation.\n    \"\"\"\n    if volume > 0:\n        return \"Ecosystem enhancement\" if cond is None or cond > 0 else UPWARD\n    if volume < 0:\n        return \"Ecosystem degradation\" if cond is None or cond < 0 else DOWNWARD\n    return None\n\n\ndef asset_account(comp, state, supply, prices_tbl, valuation, by_stratum=False, log=None,\n                  condition=None):\n    \"\"\"\n    Monetary ecosystem asset account (SEEA EA ch. 10 and Annex 10.1).\n\n    Per class c and service s, the value is NPV = P x u x a, where\n      a = class area,\n      u = annual supply per area unit (expected flow intensity),\n      P = price x annuity(r, L), i.e. today's flow is assumed to continue for\n          the asset life.\n    Both ends are valued as SEEA EA Annex 10.1 does: the value at a date t counts\n    the flows expected from year t+1 (V_t = sum over j = t+1.. ), so a closing value\n    equals the next period's opening value.\n    Opening: a = opening area, u = supply per area in year o+1, price of year o.\n    Closing: a = closing area, u = supply per area in year c+1, price of year c.\n    If the run ends at c (no year c+1), or a class has no area in c+1, the flow of\n    year c is used for the closing (logged). If a class has no area in one of the\n    two years, the other year's u is used.\n\n    The change P1u1a1 - P0u0a0 is split exactly into area, volume and price\n    effects with the symmetric weights of SEEA EA Annex 10.1:\n      area   = (P0u0/3 + P0u1/6 + P1u0/6 + P1u1/3) x (a1 - a0)\n      volume = (P0a0/3 + P0a1/6 + P1a0/6 + P1a1/3) x (u1 - u0)\n      price  = (u0a0/3 + u0a1/6 + u1a0/6 + u1a1/3) x (P1 - P0)\n    The area effect is split by the extent account into conversions\n    (additions, reductions), catastrophic losses and unexplained change; the\n    volume effect (summed over services) into enhancement, degradation or\n    reappraisals using the condition change (classify_volume); the price\n    effect is revaluation.\n    \"\"\"\n    if supply is None or prices_tbl is None or prices_tbl.empty:\n        _log(log, \"SEEA-Sim: no priced services; asset account skipped.\")\n        return None\n    if valuation is None or valuation.empty:\n        _log(log, \"SEEA-Sim: Asset Valuation is empty; asset account skipped.\")\n        return None\n    default = valuation[valuation[\"StateClassId\"].isna()]\n    per_class = valuation[valuation[\"StateClassId\"].notna()].set_index(\"StateClassId\")\n\n    def af(cls):\n        if cls in per_class.index:\n            r = per_class.loc[cls]\n        elif not default.empty:\n            r = default.iloc[0]\n        else:\n            return np.nan\n        return annuity_factor(float(r.DiscountRate), int(r.AssetLifeYears))\n\n    cond_of = condition_lookup(condition)\n    if condition is None or condition.empty:\n        _log(log, \"SEEA-Sim: no condition data; volume gains are booked as enhancement and \"\n                  \"losses as degradation (no reappraisals).\")\n\n    st = _prep_state(state, by_stratum)\n    keys = [\"Iteration\", \"StratumId\", \"StateClassId\"]\n    q = supply.merge(st, on=[\"Iteration\", \"StratumId\", \"Timestep\", \"StateClassId\"],\n                     how=\"left\", suffixes=(\"\", \"_area\"))\n    q[\"PerArea\"] = np.where(q[\"Amount_area\"] > 0, q[\"Amount\"] / q[\"Amount_area\"], np.nan)\n    prices = prices_tbl.set_index([\"ServiceId\", \"Timestep\"])[\"Price\"]\n    q = q[q.ServiceId.isin(set(prices_tbl.ServiceId))]\n\n    out = []\n    have_years = set(int(t) for t in q.Timestep.unique())\n    for (o, c), comp_p in comp.groupby(PERIOD):\n        o, c = int(o), int(c)\n        u0 = q[q.Timestep == o + 1].set_index(keys + [\"ServiceId\"])[\"PerArea\"]\n        u1_next = q[q.Timestep == c + 1].set_index(keys + [\"ServiceId\"])[\"PerArea\"]\n        u1_same = q[q.Timestep == c].set_index(keys + [\"ServiceId\"])[\"PerArea\"]\n        if (c + 1) not in have_years:\n            _log(log, f\"SEEA-Sim: no year {c + 1} in the run; the closing value at {c} uses \"\n                      f\"the flows of {c}. Run to {c + 1} for a closing that equals the next \"\n                      \"opening (SEEA EA Annex 10.1).\")\n        u1 = u1_next.combine_first(u1_same)\n        s = pd.concat([u0.rename(\"u0\"), u1.rename(\"u1\")], axis=1)\n        s[\"u0\"] = s[\"u0\"].fillna(s[\"u1\"])\n        s[\"u1\"] = s[\"u1\"].fillna(s[\"u0\"])\n        s = s.fillna(0.0).reset_index()\n        s[\"AF\"] = s.StateClassId.map(af)\n        if s.AF.isna().any():\n            miss = sorted(map(str, s.loc[s.AF.isna(), \"StateClassId\"].unique()))\n            _log(log, f\"SEEA-Sim: no discount rate/asset life for {miss}; valued at 0. \"\n                      \"Add a row with no State Class as the default.\")\n            s[\"AF\"] = s.AF.fillna(0.0)\n        s[\"P0\"] = [prices.get((sv, o), np.nan) for sv in s.ServiceId]\n        s[\"P1\"] = [prices.get((sv, c), np.nan) for sv in s.ServiceId]\n        s[[\"P0\", \"P1\"]] = s[[\"P0\", \"P1\"]].fillna(0.0).mul(s.AF, axis=0)\n        s = s.merge(comp_p[keys + [\"Opening\", \"Closing\"]], on=keys, how=\"left\").fillna(\n            {\"Opening\": 0.0, \"Closing\": 0.0})\n        P0, P1, a0, a1, U0, U1 = s.P0, s.P1, s.Opening, s.Closing, s.u0, s.u1\n        s[\"V0\"] = P0 * U0 * a0\n        s[\"V1\"] = P1 * U1 * a1\n        s[\"w_area\"] = P0 * U0 / 3 + P0 * U1 / 6 + P1 * U0 / 6 + P1 * U1 / 3\n        s[\"vol\"] = (P0 * a0 / 3 + P0 * a1 / 6 + P1 * a0 / 6 + P1 * a1 / 3) * (U1 - U0)\n        s[\"price\"] = (U0 * a0 / 3 + U0 * a1 / 6 + U1 * a0 / 6 + U1 * a1 / 3) * (P1 - P0)\n        cls = s.groupby(keys)[[\"V0\", \"V1\", \"w_area\", \"vol\", \"price\"]].sum().reset_index()\n        d = comp_p.merge(cls, on=keys, how=\"left\").fillna(\n            {\"V0\": 0.0, \"V1\": 0.0, \"w_area\": 0.0, \"vol\": 0.0, \"price\": 0.0})\n        w = d.w_area\n        vals = {e: pd.Series(0.0, index=d.index) for e in ASSET_ENTRIES}\n        vals[\"Opening value\"] = d.V0\n        vals[\"Closing value\"] = d.V1\n        vals[\"Net change in value\"] = d.V1 - d.V0\n        vals[CONVERSION_ADD] = (d.Add_Managed + d.Add_Unmanaged + d.Add_Catastrophic) * w\n        vals[CONVERSION_RED] = -(d.Red_Managed + d.Red_Unmanaged) * w\n        vals[CATASTROPHIC_LOSS] = -d.Red_Catastrophic * w\n        vals[OTHER_VOLUME] = d.Residual * w\n        vals[\"Revaluations\"] = d.price\n        for i, r in d.iterrows():\n            entry = classify_volume(r.vol, cond_of(r.StateClassId, o, c))\n            if entry:\n                vals[entry].loc[i] = r.vol\n        base = d[[\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\"]]\n        scale = max(1.0, float(np.abs(d.V0).max()), float(np.abs(d.V1).max()))\n        for e in ASSET_ENTRIES:  # floating-point noise from the decomposition -> 0\n            vals[e] = vals[e].where(np.abs(vals[e]) > 1e-12 * scale, 0.0)\n        out += [base.assign(Entry=e, Amount=vals[e].values) for e in ASSET_ENTRIES]\n    return _finish(_order(pd.concat(out, ignore_index=True), ASSET_ENTRIES))\n\n\ndef annual_values(state, supply, prices_tbl, valuation, by_stratum=False, log=None):\n    \"\"\"\n    Per (Iteration, Timestep, StratumId, StateClassId):\n      ServiceValue = sum over services of supply x price in the year  (currency per year;\n                     empty in the first simulated year, which has no flows)\n      AssetValue   = NPV at the date t, valued as the asset account does: supply per\n                     area expected in year t+1 x price of year t x annuity(r, L) x area\n                     at t (flows of year t if there is no year t+1 for the class)\n      Area, ServiceValuePerArea, AssetValuePerArea (per area unit).\n    AssetValue at a period's opening and closing years equals the asset account's\n    opening and closing values.\n    \"\"\"\n    if supply is None or prices_tbl is None or prices_tbl.empty:\n        return None\n    keys = [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\"]\n    v = supply.merge(prices_tbl, on=[\"ServiceId\", \"Timestep\"], how=\"inner\")\n    if v.empty:\n        return None\n    v[\"Value\"] = v[\"Amount\"] * v[\"Price\"]\n    serv = v.groupby(keys, as_index=False)[\"Value\"].sum().rename(columns={\"Value\": \"ServiceValue\"})\n\n    af = {}\n    if valuation is not None and not valuation.empty:\n        default = valuation[valuation[\"StateClassId\"].isna()]\n        classes = set(supply.StateClassId.unique())\n        for cls in classes:\n            row = valuation[valuation[\"StateClassId\"] == cls]\n            row = row if not row.empty else default\n            if not row.empty:\n                af[cls] = annuity_factor(float(row.iloc[0].DiscountRate), int(row.iloc[0].AssetLifeYears))\n\n    st = _prep_state(state, by_stratum).rename(columns={\"Amount\": \"Area\"})\n    q = supply[supply.ServiceId.isin(set(prices_tbl.ServiceId))].merge(st, on=keys, how=\"left\")\n    q[\"PerArea\"] = np.where(q[\"Area\"] > 0, q[\"Amount\"] / q[\"Area\"], np.nan)\n    idx = keys + [\"ServiceId\"]\n    cur = q.set_index(idx)[\"PerArea\"].dropna()\n    nxt = q.assign(Timestep=q.Timestep - 1).set_index(idx)[\"PerArea\"].dropna()\n    u = nxt.combine_first(cur).rename(\"u\").reset_index()\n    u = u.merge(prices_tbl[[\"ServiceId\", \"Timestep\", \"Price\"]], on=[\"ServiceId\", \"Timestep\"], how=\"inner\")\n    u[\"APA\"] = u[\"u\"] * u[\"Price\"] * u[\"StateClassId\"].map(af).fillna(0.0)\n    apa = u.groupby(keys, as_index=False)[\"APA\"].sum().rename(columns={\"APA\": \"AssetValuePerArea\"})\n\n    out = st[st.Area > 0].merge(serv, on=keys, how=\"outer\").merge(apa, on=keys, how=\"left\")\n    out = out[out.Timestep.isin(set(apa.Timestep) | set(serv.Timestep))]\n    out[\"Area\"] = out[\"Area\"].fillna(0.0)\n    out[\"AssetValuePerArea\"] = out[\"AssetValuePerArea\"].fillna(0.0)\n    out[\"AssetValue\"] = out[\"AssetValuePerArea\"] * out[\"Area\"]\n    out[\"ServiceValuePerArea\"] = np.where(out.Area > 0, out.ServiceValue / out.Area, np.nan)\n    out.loc[out.ServiceValue.isna(), \"ServiceValuePerArea\"] = np.nan\n    return out[keys + [\"ServiceValue\", \"AssetValue\", \"Area\", \"ServiceValuePerArea\",\n                       \"AssetValuePerArea\"]].sort_values(keys).reset_index(drop=True)\n\n\ndef account_checks(table: pd.DataFrame, account: str, opening: str, closing: str,\n                   net: str, reductions=()) -> pd.DataFrame:\n    \"\"\"Opening + additions - reductions - closing; entries listed in `reductions`\n    are positive amounts that are subtracted, all other change entries are added.\"\"\"\n    t = table.copy()\n    t[\"StratumId\"] = t[\"StratumId\"].astype(object).where(t[\"StratumId\"].notna(), ALL)\n    idx = [c for c in [\"Iteration\", \"StratumId\", *PERIOD, \"StateClassId\", \"StockGroupId\"]\n           if c in t.columns]\n    w = t.pivot_table(index=idx, columns=\"Entry\", values=\"Amount\", aggfunc=\"sum\").fillna(0.0)\n    changes = [c for c in w.columns if c not in (opening, closing, net)]\n    sign = pd.Series([-1.0 if c in reductions else 1.0 for c in changes], index=changes)\n    res = (w[opening] + (w[changes] * sign).sum(axis=1) - w[closing]).abs()\n    res = res.groupby([\"Iteration\", \"StratumId\", *PERIOD]).sum().rename(\"Residual\").reset_index()\n    return res.assign(Account=account, Item=\"Opening + changes - closing (should be 0)\")\n\n\ndef finish_checks(parts) -> pd.DataFrame:\n    parts = [p for p in parts if p is not None and not p.empty]\n    if not parts:\n        return pd.DataFrame(columns=[\"Iteration\", \"Account\", *PERIOD, \"StratumId\", \"Item\", \"Residual\"])\n    out = pd.concat(parts, ignore_index=True)\n    return _finish(out[[\"Iteration\", \"Account\", *PERIOD, \"StratumId\", \"Item\", \"Residual\"]])\n",
 "seea_outputs.py": "\"\"\"\nSEEA-Sim extra outputs: an Excel workbook with the accounts in SEEA layout,\nand value maps made from ST-Sim's state-class maps.\n\nNo SyncroSim import; transformer.py passes DataFrames and file paths.\n\"\"\"\nfrom __future__ import annotations\n\nimport os\n\nimport numpy as np\nimport pandas as pd\n\nimport seea_accounts as sa\n\nNODATA = -9999.0\n\n\n# ---------------------------------------------------------------- workbook\n\ndef _mean_table(df, rows, cols, value=\"Amount\", row_order=None):\n    \"\"\"Mean over iterations of a long table, rows x cols, with a Total column.\n    row_order orders the last row level (e.g. the SEEA entry order).\"\"\"\n    if df is None or df.empty:\n        return None\n    n_it = df[\"Iteration\"].nunique()\n    t = (df.groupby(rows + cols, dropna=False)[value].sum() / n_it).unstack(cols).fillna(0.0)\n    if len(cols) == 1:\n        t[\"Total\"] = t.sum(axis=1)\n    if row_order is not None:\n        t = t.reset_index()\n        last = rows[-1]\n        t[\"_o\"] = t[last].map({e: i for i, e in enumerate(row_order)}).fillna(len(row_order))\n        t = t.sort_values(rows[:-1] + [\"_o\"]).drop(columns=\"_o\").set_index(rows)\n    return t\n\n\ndef _str(df):\n    \"\"\"Ids may be names or numbers; Excel sheets want plain values.\"\"\"\n    if df is None:\n        return None\n    df = df.copy()\n    for c in df.columns:\n        if df[c].dtype == object:\n            df[c] = df[c].where(df[c].isna(), df[c].astype(str))\n    return df\n\n\ndef template(entries, drop=()):\n    \"\"\"UN row layout from 'Group: item' entry names: a bold group header row,\n    then its items indented. Returns [(label, entry or None, indent)].\"\"\"\n    rows, group = [], None\n    for e in entries:\n        if e in drop:\n            continue\n        if \": \" in e:\n            g, item = e.split(\": \", 1)\n            if g != group:\n                rows.append((g, None, 0))\n                group = g\n            rows.append((item, e, 1))\n        else:\n            group = None\n            rows.append((e, e, 0))\n    return rows\n\n\nSERVICE_GROUPS = {1: \"Provisioning services\", 2: \"Regulating and maintenance services\",\n                  3: \"Cultural services\"}\n\n\ndef _service_group(v):\n    if v is None or (isinstance(v, float) and np.isnan(v)):\n        return None\n    try:\n        return SERVICE_GROUPS[int(float(v))]\n    except (ValueError, KeyError):\n        t = str(v).strip().lower()\n        for g in SERVICE_GROUPS.values():\n            if g.lower().startswith(t):\n                return g\n        return str(v)\n\n\ndef _blocks(df):\n    \"\"\"Split a long table by stratum (if reported by stratum) and accounting period.\"\"\"\n    df = df.copy()\n    if \"StratumId\" not in df.columns:\n        df[\"StratumId\"] = None\n    df[\"_s\"] = df[\"StratumId\"].astype(object).where(df[\"StratumId\"].notna(), \"\")\n    for (s, o, c), b in df.groupby([\"_s\", \"OpeningTimestep\", \"ClosingTimestep\"], sort=True):\n        yield (f\", {s}\" if s else \"\"), int(o), int(c), b\n\n\nclass _Sheet:\n    \"\"\"Writes UN-style tables one under the other on an openpyxl worksheet.\"\"\"\n\n    def __init__(self, wb, name):\n        from openpyxl.styles import Alignment, Font\n        self.ws = wb.create_sheet(name[:31])\n        self.r = 1\n        self.bold, self.Alignment, self.Font = Font(bold=True), Alignment, Font\n        self.ws.column_dimensions[\"A\"].width = 46\n\n    def table(self, title, corner, cols, rows, values, total=None, extra=None, note=None,\n              fmt=\"#,##0.00\", width=16):\n        \"\"\"\n        rows: [(label, key or None, indent)]; values: DataFrame index=key, columns=cols.\n        total: label of a right-hand total column (sum over cols), or None.\n        extra: (header, {key: text}) for a text column before the value columns.\n        \"\"\"\n        from openpyxl.utils import get_column_letter\n        ws = self.ws\n        ws.cell(self.r, 1, title).font = self.Font(bold=True, size=12)\n        self.r += 1\n        if note:\n            ws.cell(self.r, 1, note).font = self.Font(italic=True)\n            self.r += 1\n        head = [corner] + ([extra[0]] if extra else []) + [str(c) for c in cols] + ([total] if total else [])\n        for j, h in enumerate(head, 1):\n            cell = ws.cell(self.r, j, h)\n            cell.font = self.bold\n            cell.alignment = self.Alignment(wrap_text=True, horizontal=\"left\" if j == 1 else \"center\")\n            if j > 1:\n                ws.column_dimensions[get_column_letter(j)].width = max(\n                    ws.column_dimensions[get_column_letter(j)].width or 0, width)\n        self.r += 1\n        off = 2 + (1 if extra else 0)\n        for label, key, indent in rows:\n            cell = ws.cell(self.r, 1, label)\n            cell.alignment = self.Alignment(horizontal=\"left\", indent=2 * indent)\n            if key is None:\n                cell.font = self.bold\n            else:\n                if extra:\n                    ws.cell(self.r, 2, extra[1].get(key, \"\"))\n                vals = [float(values.at[key, c]) if key in values.index and c in values.columns else 0.0\n                        for c in cols]\n                if total:\n                    vals.append(sum(vals))\n                for j, v in enumerate(vals):\n                    ws.cell(self.r, off + j, v).number_format = fmt\n            self.r += 1\n        self.r += 1\n\n\ndef _means(df, index, col=\"StateClassId\", value=\"Amount\", n_it=1):\n    t = (df.groupby([index, col], dropna=False)[value].sum() / n_it).unstack(col).fillna(0.0)\n    return t\n\n\ndef _classes(*dfs, cols=(\"StateClassId\",)):\n    s = set()\n    for d in dfs:\n        if d is not None:\n            for c in cols:\n                if c in d.columns:\n                    s |= set(d[c].dropna().astype(str))\n    return sorted(s)\n\n\ndef write_workbook(path, tables: dict, info: dict):\n    \"\"\"\n    tables: name -> long DataFrame as saved to SyncroSim (may be None).\n    info:   scenario, version, iterations, currency, services (Service sheet), periods.\n\n    Sheets in the UN SEEA EA (2024) layout, means over iterations:\n      Extent account (Table 4.1 / AI.2), Change matrix (Table 4.2 / AI.3),\n      Carbon stock account (Table 13.3), Supply physical (Table 7.1a),\n      Supply monetary (Table 9.1a), Asset account (Table 10.1 / AI.11).\n    Then SEEA-Sim detail (supply by year, asset value by year, carbon by process,\n    carbon flows, checks) and the raw per-iteration tables.\n    \"\"\"\n    from openpyxl import Workbook\n\n    extent = _str(tables.get(\"OutputExtent\"))\n    matrix = _str(tables.get(\"OutputChangeMatrix\"))\n    caccount = _str(tables.get(\"OutputCarbonAccount\"))\n    cstock = _str(tables.get(\"OutputCarbonStock\"))\n    cflow = _str(tables.get(\"OutputCarbonFlow\"))\n    phys = _str(tables.get(\"OutputServiceSupply\"))\n    mon = _str(tables.get(\"OutputServiceValue\"))\n    asset = _str(tables.get(\"OutputAssetAccount\"))\n    annual = _str(tables.get(\"OutputAssetValue\"))\n    checks = _str(tables.get(\"OutputCheck\"))\n    currency = info.get(\"currency\") or \"currency units\"\n\n    n_it = max(1, int(info.get(\"iterations\") or 1))\n    for d in (extent, matrix, caccount, phys, mon, asset):\n        if d is not None and not d.empty:\n            n_it = d[\"Iteration\"].nunique()\n            break\n    classes = _classes(extent, asset, phys, mon, caccount) or _classes(matrix, cols=(\"FromStateClassId\", \"ToStateClassId\"))\n\n    wb = Workbook()\n    about = wb.active\n    about.title = \"About\"\n    notes = [\n        \"SEEA-Sim accounts, \" + str(info.get(\"scenario\", \"\")),\n        f\"SEEA-Sim {info.get('version', '')}; {info.get('iterations', '?')} iterations. \"\n        \"The UN-layout sheets show means over iterations; the 'raw' sheets hold every iteration.\",\n        \"Areas in the ST-Sim amount unit (Terminology); carbon in the ST-Sim stock unit; \"\n        f\"money in {currency}.\",\n        \"Layout follows UN SEEA EA (2024): Extent account = Table 4.1/AI.2; Change matrix = Table 4.2/AI.3; \"\n        \"Carbon stock account = Table 13.3; Supply = Tables 7.1a and 9.1a; Asset account = Table 10.1/AI.11.\",\n        \"Signs as in the UN tables: extent and carbon reductions are positive amounts (net change = additions \"\n        \"minus reductions); in the monetary asset account decreases are negative (net change = sum of entries).\",\n        \"Change matrix: off-diagonal cells are gross transitions summed over the period's timesteps; the UN \"\n        \"compares the opening and closing maps. They differ only where a cell changed more than once in the period.\",\n        \"Carbon stock account: columns are ecosystem types (biocarbon in ecosystems). Flows triggered by \"\n        \"transition types classed Managed are managed expansion/contraction, Catastrophic ones catastrophic \"\n        \"losses, all others unmanaged. Reclassifications = carbon that moved with converted area (net per \"\n        \"type; they cancel over all types). Discoveries, imports and exports are geocarbon/economy entries (0).\",\n        \"Supply tables cover each accounting period: the sum of years opening+1 to closing. 'by year' sheets \"\n        \"show each year. Physical supply is not totalled over services (different units, SEEA EA 2.78).\",\n        \"Ecosystem services are grouped by the Service sheet's Type; units from its Physical Units column.\",\n    ]\n    for i, n in enumerate(notes, 1):\n        about.cell(i, 1, n)\n    about.column_dimensions[\"A\"].width = 140\n\n    # Extent account\n    if extent is not None and not extent.empty:\n        sh = _Sheet(wb, \"Extent account\")\n        for strat, o, c, b in _blocks(extent):\n            sh.table(f\"Ecosystem extent account, {o}-{c}{strat} (area units)\", \"Accounting entries\",\n                     classes, template(sa.EXTENT_ENTRIES), _means(b, \"Entry\", n_it=n_it), total=\"TOTAL\")\n\n    # Change matrix\n    if matrix is not None and not matrix.empty:\n        sh = _Sheet(wb, \"Change matrix\")\n        for strat, o, c, b in _blocks(matrix):\n            m = _means(b.rename(columns={\"ToStateClassId\": \"To\"}), \"FromStateClassId\", col=\"To\", n_it=n_it)\n            m = m.reindex(index=classes, columns=classes).fillna(0.0)\n            m[\"Opening extent\"] = m.sum(axis=1)\n            m.loc[\"Closing extent\"] = m.sum(axis=0)\n            rows = [(k, k, 0) for k in classes] + [(\"Closing extent\", \"Closing extent\", 0)]\n            sh.table(f\"Ecosystem type change matrix, {o}-{c}{strat} (area units)\",\n                     \"Ecosystem types: opening (rows), closing (columns)\", classes + [\"Opening extent\"], rows, m)\n\n    # Carbon stock account (Table 13.3)\n    if caccount is not None and not caccount.empty:\n        sh = _Sheet(wb, \"Carbon stock account\")\n        for strat, o, c, b in _blocks(caccount):\n            parts = [(\"all ecosystem carbon pools\", b)] + [\n                (f\"pool {g}\", bg) for g, bg in b.groupby(\"StockGroupId\", sort=True)]\n            for what, bb in parts:\n                sh.table(f\"Carbon stock account, {o}-{c}{strat}: biocarbon in ecosystems, {what} \"\n                         \"(carbon units)\", \"Accounting entries\", classes, template(sa.CARBON_ENTRIES),\n                         _means(bb, \"Entry\", n_it=n_it), total=\"Total\")\n\n    # Supply tables\n    svc = info.get(\"services\")\n    unit, group = {}, {}\n    if svc is not None and not getattr(svc, \"empty\", True):\n        for _, r in svc.iterrows():\n            unit[str(r.get(\"Name\"))] = \"\" if pd.isna(r.get(\"Units\")) else str(r.get(\"Units\"))\n            group[str(r.get(\"Name\"))] = _service_group(r.get(\"ServiceType\"))\n\n    def service_rows(names, monetary):\n        order = list(SERVICE_GROUPS.values())\n        keyed = sorted(names, key=lambda n: (order.index(group[n]) if group.get(n) in order else len(order),\n                                             str(group.get(n) or \"\"), n))\n        rows, g0 = [], object()\n        for n in keyed:\n            g = group.get(n)\n            if g and g != g0:\n                rows.append((g, None, 0))\n            rows.append((n, n, 1 if g else 0))\n            g0 = g\n        if monetary:\n            rows.append((\"Total ecosystem services\", \"__total__\", 0))\n        return rows\n\n    periods = info.get(\"periods\")\n    if not periods:\n        src = next((d for d in (extent, asset, caccount) if d is not None and not d.empty), None)\n        periods = [] if src is None else sorted(\n            {(int(o), int(c)) for o, c in src[[\"OpeningTimestep\", \"ClosingTimestep\"]].itertuples(index=False)})\n    for df, monetary, name, tbl in ((phys, False, \"Supply physical\", \"7.1a\"), (mon, True, \"Supply monetary\", \"9.1a\")):\n        if df is None or df.empty:\n            continue\n        what = (f\"Ecosystem services supply table in monetary terms ({currency})\" if monetary\n                else \"Ecosystem services supply table in physical terms\")\n        names = sorted(df[\"ServiceId\"].dropna().unique())\n        sdf = df.copy()\n        if \"StratumId\" not in sdf.columns:\n            sdf[\"StratumId\"] = None\n        sdf[\"_s\"] = sdf[\"StratumId\"].astype(object).where(sdf[\"StratumId\"].notna(), \"\")\n\n        def put(sh, title, part):\n            v = _means(part, \"ServiceId\", n_it=n_it)\n            if monetary:\n                v.loc[\"__total__\"] = v.sum(axis=0)\n            sh.table(title, \"Ecosystem services\", classes, service_rows(names, monetary), v,\n                     total=\"Total supply by ecosystem assets\",\n                     extra=None if monetary else (\"Measurement unit\", unit),\n                     fmt=\"#,##0\" if monetary else \"#,##0.00\", width=22 if monetary else 16)\n\n        sh = _Sheet(wb, name)\n        for s_, part_s in sdf.groupby(\"_s\", sort=True):\n            strat = f\", {s_}\" if s_ else \"\"\n            for o, c in periods:\n                part = part_s[(part_s.Timestep > o) & (part_s.Timestep <= c)]\n                if not part.empty:\n                    put(sh, f\"{what}, {o}-{c}{strat} (sum of years {o + 1}-{c}; SEEA EA Table {tbl})\", part)\n        sh = _Sheet(wb, f\"{name} by year\")\n        for s_, part_s in sdf.groupby(\"_s\", sort=True):\n            strat = f\", {s_}\" if s_ else \"\"\n            for t, part in part_s.groupby(\"Timestep\", sort=True):\n                put(sh, f\"{what}, {int(t)}{strat}\", part)\n\n    # Monetary asset account (Table 10.1 / AI.11)\n    if asset is not None and not asset.empty:\n        drop = () if (asset.loc[asset.Entry == sa.OTHER_VOLUME, \"Amount\"].abs() > 1e-9).any() else (sa.OTHER_VOLUME,)\n        sh = _Sheet(wb, \"Asset account\")\n        for strat, o, c, b in _blocks(asset):\n            sh.table(f\"Monetary ecosystem asset account, {o}-{c}{strat} ({currency})\", \"Accounting entries\",\n                     classes, template(sa.ASSET_ENTRIES, drop=drop), _means(b, \"Entry\", n_it=n_it),\n                     total=\"TOTAL\", fmt=\"#,##0\", width=22)\n\n    # SEEA-Sim detail and raw tables (plain pandas layout)\n    detail = []\n    if annual is not None:\n        detail.append((\"Asset value by year\", _mean_table(annual, [\"Timestep\"], [\"StateClassId\"], value=\"AssetValue\")))\n    if cstock is not None:\n        detail.append((\"Carbon by process\", _mean_table(cstock, [\"OpeningTimestep\", \"ClosingTimestep\", \"StockGroupId\",\n                                                                 \"Entry\"], [\"StateClassId\"],\n                                                        row_order=sa.STOCK_ENTRIES)))\n    if cflow is not None:\n        detail.append((\"Carbon flows\", _mean_table(cflow, [\"OpeningTimestep\", \"ClosingTimestep\", \"Category\"],\n                                                   [\"StateClassId\"])))\n    if checks is not None and not checks.empty:\n        detail.append((\"Checks\", checks.groupby([\"Account\", \"Item\"])[\"Residual\"].max()\n                       .rename(\"Max residual over iterations\").to_frame()))\n    raw = ((\"raw Extent\", extent), (\"raw Change matrix\", matrix), (\"raw Carbon account\", caccount),\n           (\"raw Carbon by process\", cstock), (\"raw Carbon flows\", cflow), (\"raw Supply physical\", phys),\n           (\"raw Supply monetary\", mon), (\"raw Asset account\", asset), (\"raw Asset value by year\", annual),\n           (\"raw Checks\", checks))\n    import tempfile\n    with tempfile.TemporaryDirectory() as tmp:\n        side = os.path.join(tmp, \"detail.xlsx\")\n        with pd.ExcelWriter(side, engine=\"openpyxl\") as xw:\n            pd.DataFrame({\"\": [\"\"]}).to_excel(xw, sheet_name=\"_\", index=False)\n            for name, t in detail:\n                if t is not None and not t.empty:\n                    t.to_excel(xw, sheet_name=name[:31])\n            for name, t in raw:\n                if t is not None and not t.empty:\n                    t.to_excel(xw, sheet_name=name[:31], index=False)\n        _copy_sheets(side, wb, skip=(\"_\",))\n    wb.save(path)\n    return path\n\n\ndef _copy_sheets(src_path, wb, skip=()):\n    \"\"\"Append the values of every sheet of src_path to workbook wb.\"\"\"\n    from openpyxl import load_workbook\n    src = load_workbook(src_path)\n    for ws in src.worksheets:\n        if ws.title in skip:\n            continue\n        dst = wb.create_sheet(ws.title)\n        for row in ws.iter_rows(values_only=True):\n            dst.append(list(row))\n\n\n# ---------------------------------------------------------------- maps\n\ndef class_codes(stateclass: pd.DataFrame) -> dict:\n    \"\"\"stsim_StateClass -> {raster value: state class name}. The raster holds the 'Id' column.\"\"\"\n    idcol = next((c for c in stateclass.columns if c.lower() == \"id\"), None)\n    if idcol is None:\n        raise sa.AccountError(f\"stsim_StateClass has no Id column; columns {list(stateclass.columns)}\")\n    d = stateclass.dropna(subset=[idcol])\n    return {int(r[idcol]): r[\"Name\"] for _, r in d.iterrows()}\n\n\ndef map_years(periods) -> list:\n    \"\"\"Years to map: every accounting period boundary (opening and closing years).\n    With one period per year this gives a map for every year.\"\"\"\n    return sorted({int(y) for p in periods for y in p})\n\n\ndef per_area_for_years(per_area: pd.DataFrame, years) -> pd.DataFrame:\n    \"\"\"Per-area values for each map year, column by column. AssetValuePerArea is the\n    value at that date (flows expected from the next year). ServiceValuePerArea is the\n    flow in that year; the first simulated year has none, so it takes the next year\n    that has one, as the asset account does for its opening value.\"\"\"\n    keys = [\"Iteration\", \"StateClassId\"]\n    parts = []\n    for y in years:\n        cols = {}\n        for col in (\"ServiceValuePerArea\", \"AssetValuePerArea\"):\n            have = sorted(int(t) for t in per_area.loc[per_area[col].notna(), \"Timestep\"].unique())\n            src = y if y in have else next((t for t in have if t > y), None)\n            if src is not None:\n                cols[col] = per_area.loc[(per_area.Timestep == src) & per_area[col].notna(),\n                                         keys + [col]]\n        if not cols:\n            continue\n        part = None\n        for d in cols.values():\n            part = d if part is None else part.merge(d, on=keys, how=\"outer\")\n        part = part.assign(Timestep=y)\n        for col in (\"ServiceValuePerArea\", \"AssetValuePerArea\"):\n            if col not in part:\n                part[col] = 0.0\n        parts.append(part.fillna(0.0))\n    if not parts:\n        return per_area.iloc[0:0]\n    return pd.concat(parts, ignore_index=True)[[\"Iteration\", \"Timestep\", \"StateClassId\",\n                                                \"ServiceValuePerArea\", \"AssetValuePerArea\"]]\n\n\ndef write_value_maps(state_maps: pd.DataFrame, codes: dict, per_area: pd.DataFrame,\n                     timesteps, out_dir: str, log=None):\n    \"\"\"\n    state_maps: stsim_OutputSpatialState with full paths (Iteration, Timestep, Filename[, Band]).\n    per_area:   annual_values(..., by_stratum=False) (Iteration, Timestep, StateClassId,\n                ServiceValuePerArea, AssetValuePerArea).\n    Writes two float32 GeoTIFFs per iteration and timestep: service value per area unit per\n    year, and asset value per area unit. Returns rows for OutputSpatialValue.\n    \"\"\"\n    import rasterio\n\n    os.makedirs(out_dir, exist_ok=True)\n    rows = []\n    maps = state_maps[state_maps[\"Timestep\"].isin(timesteps)]\n    if maps.empty:\n        sa._log(log, \"SEEA-Sim: no ST-Sim state-class maps at the accounting years \"\n                     f\"{sorted(timesteps)}; value maps skipped. Turn on Output Options > Spatial > \"\n                     \"State classes for those years.\")\n        return rows\n    for _, m in maps.iterrows():\n        it, ts = int(m[\"Iteration\"]), int(m[\"Timestep\"])\n        pa = per_area[(per_area.Iteration == it) & (per_area.Timestep == ts)].set_index(\"StateClassId\")\n        band = int(m[\"Band\"]) if \"Band\" in m and pd.notna(m[\"Band\"]) else 1\n        with rasterio.open(m[\"Filename\"]) as src:\n            cls = src.read(band)\n            nod = src.nodata\n            profile = src.profile.copy()\n        profile.update(dtype=\"float32\", count=1, nodata=NODATA, compress=\"deflate\")\n        valid = np.ones(cls.shape, bool) if nod is None else (cls != nod)\n        lut_max = int(max(max(codes), int(cls[valid].max()) if valid.any() else 0)) + 1\n        out = {}\n        for col, tag in ((\"ServiceValuePerArea\", \"value\"), (\"AssetValuePerArea\", \"asset\")):\n            lut = np.full(lut_max, NODATA, dtype=\"float32\")\n            for code, name in codes.items():\n                if code < lut_max:\n                    lut[code] = float(pa[col].get(name, 0.0)) if name in pa.index else 0.0\n            arr = np.full(cls.shape, NODATA, dtype=\"float32\")\n            ok = valid & (cls >= 0) & (cls < lut_max)\n            arr[ok] = lut[cls[ok].astype(int)]\n            fn = os.path.join(out_dir, f\"seeasim_{tag}_it{it}_ts{ts}.tif\")\n            with rasterio.open(fn, \"w\", **profile) as dst:\n                dst.write(arr, 1)\n            out[tag] = fn\n        rows.append({\"Iteration\": it, \"Timestep\": ts,\n                     \"ServiceValueFile\": out[\"value\"], \"AssetValueFile\": out[\"asset\"]})\n    return rows\n",
 "stsimseea.yml": "name: stsimseea\nchannels:\n  - conda-forge\n  - nodefaults\ndependencies:\n  - python=3.11\n  - pandas>=2.2\n  - numpy\n  - pysyncrosim>=2.1\n  - rasterio\n  - openpyxl\n",
 "transformer.py": "\"\"\"\nSEEA-Sim transformer: reads ST-Sim output from the running result scenario,\nbuilds the SEEA EA accounts (seea_accounts.py), saves them, and writes the\nextra outputs (seea_outputs.py): yearly asset values for the Charts tab,\nvalue maps for the Maps tab, and an Excel workbook.\n\nRuns as the second pipeline stage, after stsim_Runtime.\n\"\"\"\nimport os\nimport re\nimport sys\nimport tempfile\nimport traceback\n\nimport pandas as pd\nimport pysyncrosim as ps\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport seea_accounts as sa  # noqa: E402\n\nPKG = \"stsimseea\"\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n\ndef say(msg, kind=\"status\"):\n    \"\"\"Write to the SyncroSim Run Log (same protocol as ps.update_run_log).\"\"\"\n    prefix = {\"status\": \"ssim-task-log\", \"info\": \"ssim-task-info\", \"warning\": \"ssim-task-warning\"}[kind]\n    for i, line in enumerate(str(msg).split(\"\\n\")):\n        print(f\"{prefix if i == 0 else 'ssim-task-log'}={line}\\r\", flush=True)\n\n\ndef warn(msg):\n    say(msg, \"warning\")\n\n\ndef version():\n    try:\n        xml = open(os.path.join(HERE, \"package.xml\"), encoding=\"utf-8-sig\").read()\n        return re.search(r'<package\\s[^>]*version=\"([^\"]+)\"', xml).group(1)\n    except Exception:\n        return \"\"\n\n\ndef read(scn, name, empty_ok=True, **kw):\n    try:\n        df = scn.datasheets(name=name, **kw)\n    except Exception as e:  # sheet missing from this library\n        if empty_ok:\n            return pd.DataFrame()\n        raise sa.AccountError(f\"Could not read {name}: {e}\")\n    if (df is None or df.empty) and not empty_ok:\n        raise sa.AccountError(f\"{name} is empty. Check that ST-Sim wrote this output.\")\n    return df if df is not None else pd.DataFrame()\n\n\ndef state_class_table(scn):\n    \"\"\"stsim_StateClass with its Id (raster code) column. It is a project-level datasheet, so\n    reading it through the scenario can come back empty; fall back to the project, then to the\n    library file itself.\"\"\"\n    def has_id(d):\n        return d is not None and not d.empty and any(str(c).lower() == \"id\" for c in d.columns)\n\n    for get in (lambda: scn.datasheets(name=\"stsim_StateClass\"),\n                lambda: scn.project.datasheets(name=\"stsim_StateClass\", return_hidden=True)):\n        try:\n            d = get()\n            if has_id(d):\n                return d\n        except Exception:\n            pass\n    import sqlite3\n    with sqlite3.connect(scn.library.location) as con:\n        d = pd.read_sql_query(\"SELECT Id, Name FROM stsim_StateClass WHERE ProjectId = ?\",\n                              con, params=(int(scn.project_id),))\n    if not has_id(d):\n        raise sa.AccountError(\"stsim_StateClass has no Id values in this project\")\n    return d\n\n\ndef save(scn, name, df, columns):\n    if df is None or df.empty:\n        return\n    # SyncroSim 3.2 imports a CSV's rows in reverse order (checked with SyncroSim.Console\n    # --import); hand them over reversed so the library keeps the UN row order.\n    data = df[[c for c in columns if c in df.columns]].iloc[::-1].reset_index(drop=True)\n    scn.save_datasheet(name=f\"{PKG}_{name}\", data=data)\n    say(f\"SEEA-Sim: saved {name} ({len(df)} rows)\")\n\n\ndef temp_folder(name):\n    try:\n        return ps.runtime_temp_folder(name)\n    except Exception:\n        return tempfile.mkdtemp(prefix=name + \"-\")\n\n\ndef check_output_options(scn):\n    \"\"\"Fail early with a clear message if ST-Sim did not write what we need.\"\"\"\n    oo = read(scn, \"stsim_OutputOptions\")\n    if oo.empty:\n        return\n    r = oo.iloc[0]\n    if not bool(r.get(\"SummaryOutputTRSC\", False)):\n        raise sa.AccountError(\n            \"Turn on ST-Sim Output Options > Tabular > Transitions by state class \"\n            \"(SummaryOutputTRSC), every 1 timestep. The extent account is built from it.\")\n    if pd.notna(r.get(\"SummaryOutputTRSCTimesteps\")) and int(r[\"SummaryOutputTRSCTimesteps\"]) != 1:\n        raise sa.AccountError(\"Set the Transitions by state class output frequency to 1 timestep.\")\n\n\ndef flag(opts, col, default=True):\n    if opts.empty or col not in opts.columns or pd.isna(opts.iloc[0].get(col)):\n        return default\n    v = opts.iloc[0][col]\n    return str(v).strip().lower() in (\"true\", \"yes\", \"1\") if isinstance(v, str) else bool(v)\n\n\ndef services(scn):\n    \"\"\"Project Service sheet (Name, ServiceType, Units) for the supply tables' unit column\n    and service-type groups. Empty if it cannot be read.\"\"\"\n    df = read(scn, f\"{PKG}_Service\")\n    if df.empty:\n        try:\n            df = read(scn.project, f\"{PKG}_Service\")\n        except Exception:\n            df = pd.DataFrame()\n    return df\n\n\ndef main():\n    scn = ps.Scenario()\n    say(f\"SEEA-Sim {version()}: building accounts\")\n\n    opts = read(scn, f\"{PKG}_Options\")\n    by_stratum = flag(opts, \"ReportByStratum\", False)\n    currency = str(opts.iloc[0].get(\"Currency\")) if not opts.empty and pd.notna(opts.iloc[0].get(\"Currency\")) else \"\"\n\n    check_output_options(scn)\n    state = read(scn, \"stsim_OutputStratumState\", empty_ok=False)\n    trans = read(scn, \"stsim_OutputStratumTransitionState\")\n    if trans.empty:\n        trans = pd.DataFrame(columns=[\"Iteration\", \"Timestep\", \"StratumId\", \"TransitionTypeId\",\n                                      \"StateClassId\", \"EndStateClassId\", \"Amount\"])\n\n    per = read(scn, f\"{PKG}_AccountingPeriod\")\n    periods = ([(int(r.OpeningTimestep), int(r.ClosingTimestep)) for r in per.itertuples()]\n               if not per.empty else sa.default_periods(state))\n    for o, c in periods:\n        if c <= o:\n            raise sa.AccountError(f\"Accounting period {o}-{c}: closing must be after opening.\")\n    say(f\"SEEA-Sim: periods {periods}; iterations {state['Iteration'].nunique()}; by stratum = {by_stratum}\")\n\n    ttc = read(scn, f\"{PKG}_TransitionTypeClass\")\n    tt_class = ({r.TransitionTypeId: sa.list_value(r.ChangeType, sa.CHANGE_TYPES, \"change type\")\n                 for r in ttc.itertuples()} if not ttc.empty else {})\n\n    tables = {}\n    # Extent account and change matrix\n    comp = sa.extent_components(state, trans, periods, tt_class, by_stratum, log=warn)\n    extent = sa.extent_account(comp)\n    matrix, matrix_chk = sa.change_matrix(state, trans, periods, tt_class, by_stratum)\n    period_cols = [\"OpeningTimestep\", \"ClosingTimestep\", \"StratumId\"]\n    save(scn, \"OutputExtent\", extent, [\"Iteration\", *period_cols, \"StateClassId\", \"Entry\", \"Amount\"])\n    save(scn, \"OutputChangeMatrix\", matrix,\n         [\"Iteration\", *period_cols, \"FromStateClassId\", \"ToStateClassId\", \"Amount\"])\n    tables.update(OutputExtent=extent, OutputChangeMatrix=matrix)\n    checks = [sa.account_checks(extent, \"Extent\", \"Opening extent\", \"Closing extent\",\n                                \"Net change in extent\", sa.EXTENT_REDUCTIONS),\n              comp.assign(Residual=comp.Residual.abs())\n                  .groupby([\"Iteration\", \"StratumId\", \"OpeningTimestep\", \"ClosingTimestep\"],\n                           as_index=False)[\"Residual\"].sum()\n                  .assign(Account=\"Extent\",\n                          Item=\"Area change not explained by recorded transitions\"),\n              matrix_chk]\n\n    # Carbon\n    stock = read(scn, \"stsim_OutputStock\")\n    flow = read(scn, \"stsim_OutputFlow\")\n    groups = read(scn, f\"{PKG}_CarbonStockGroup\")\n    if not groups.empty:\n        if stock.empty or flow.empty:\n            raise sa.AccountError(\n                \"Carbon Stock Groups are set but stsim_OutputStock or stsim_OutputFlow is empty. \"\n                \"Turn on Stock-Flow summary output (stocks and flows, every timestep).\")\n        membership = read(scn, \"stsim_StockTypeGroupMembership\")\n        if membership.empty:\n            membership = pd.DataFrame(columns=[\"StockTypeId\", \"StockGroupId\"])\n        fgroups = list(read(scn, f\"{PKG}_CarbonFlowGroup\").get(\"FlowGroupId\", []))\n        fmember = read(scn, \"stsim_FlowTypeGroupMembership\")\n        cstock, cflow, cchk, caccount = sa.carbon_accounts(\n            stock, flow, periods, groups, membership, fgroups, fmember, by_stratum, log=warn,\n            tt_class=tt_class, with_un=True)\n        save(scn, \"OutputCarbonAccount\", caccount,\n             [\"Iteration\", *period_cols, \"StateClassId\", \"StockGroupId\", \"Entry\", \"Amount\"])\n        save(scn, \"OutputCarbonStock\", cstock,\n             [\"Iteration\", *period_cols, \"StateClassId\", \"StockGroupId\", \"Entry\", \"Amount\"])\n        save(scn, \"OutputCarbonFlow\", cflow,\n             [\"Iteration\", *period_cols, \"StateClassId\", \"TransitionTypeId\", \"FromStockTypeId\",\n              \"ToStockTypeId\", \"Category\", \"Amount\"])\n        tables.update(OutputCarbonAccount=caccount, OutputCarbonStock=cstock, OutputCarbonFlow=cflow)\n        checks += [cchk, sa.account_checks(caccount, \"Carbon stock\", \"Opening stock\",\n                                           \"Closing stock\", \"Net carbon balance\",\n                                           sa.CARBON_REDUCTIONS)]\n    else:\n        say(\"SEEA-Sim: no Carbon Stock Groups; carbon accounts skipped.\")\n\n    # Ecosystem services and assets\n    rules = read(scn, f\"{PKG}_ServiceSupply\")\n    supply = sa.service_supply(state, stock if not stock.empty else None,\n                               flow if not flow.empty else None, rules, by_stratum, log=warn)\n    ptbl, valuation = None, read(scn, f\"{PKG}_AssetValuation\")\n    if supply is not None:\n        out_cols = [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"ServiceId\", \"Amount\"]\n        phys = sa._finish(supply.copy())\n        save(scn, \"OutputServiceSupply\", phys, out_cols)\n        tables[\"OutputServiceSupply\"] = phys\n        prices = read(scn, f\"{PKG}_ServicePrice\")\n        if not prices.empty and \"Timestep\" not in prices.columns:\n            prices[\"Timestep\"] = pd.NA\n        price_ts = sorted(set(supply.Timestep.unique()) | {o for o, _ in periods})\n        ptbl = sa.price_table(prices, supply.ServiceId.unique(), price_ts, log=warn)\n        value = sa.service_value(supply, ptbl)\n        if value is not None:\n            value = sa._finish(value.copy())\n            save(scn, \"OutputServiceValue\", value, out_cols)\n            tables[\"OutputServiceValue\"] = value\n        cond = read(scn, f\"{PKG}_ConditionChange\")\n        assets = sa.asset_account(comp, state, supply, ptbl, valuation, by_stratum, log=say,\n                                  condition=cond if not cond.empty else None)\n        if assets is not None:\n            save(scn, \"OutputAssetAccount\", assets,\n                 [\"Iteration\", *period_cols, \"StateClassId\", \"Entry\", \"Amount\"])\n            tables[\"OutputAssetAccount\"] = assets\n            checks.append(sa.account_checks(assets, \"Asset\", \"Opening value\", \"Closing value\",\n                                            \"Net change in value\"))\n        annual = sa.annual_values(state, supply, ptbl, valuation, by_stratum, log=warn)\n        if annual is not None:\n            annual = sa._finish(annual)\n            save(scn, \"OutputAssetValue\", annual,\n                 [\"Iteration\", \"Timestep\", \"StratumId\", \"StateClassId\", \"ServiceValue\", \"AssetValue\"])\n            tables[\"OutputAssetValue\"] = annual\n    else:\n        say(\"SEEA-Sim: no Service Supply rules; service and asset accounts skipped.\")\n\n    chk = sa.finish_checks(checks)\n    save(scn, \"OutputCheck\", chk, [\"Iteration\", \"Account\", *period_cols, \"Item\", \"Residual\"])\n    tables[\"OutputCheck\"] = chk\n    if not chk.empty:\n        for (acc, item), v in chk.groupby([\"Account\", \"Item\"])[\"Residual\"].max().items():\n            (say if abs(v) < 1e-3 else warn)(f\"SEEA-Sim check: {acc}: {item}: max {v:.6g}\")\n\n    # Value maps (Maps tab)\n    if supply is not None and ptbl is not None and flag(opts, \"MapOutput\", True):\n        try:\n            import seea_outputs as so\n            per_area = sa.annual_values(state, supply if not by_stratum else\n                                        sa.service_supply(state, stock if not stock.empty else None,\n                                                          flow if not flow.empty else None, rules, False),\n                                        ptbl, valuation, False)\n            maps = read(scn, \"stsim_OutputSpatialState\", show_full_paths=True)\n            if per_area is None or maps.empty:\n                say(\"SEEA-Sim: no ST-Sim state-class maps or no values; value maps skipped.\")\n            else:\n                codes = so.class_codes(state_class_table(scn))\n                years = so.map_years(periods)\n                rows = so.write_value_maps(maps, codes, so.per_area_for_years(per_area, years), years,\n                                           temp_folder(\"SEEASimMaps\"), log=warn)\n                if rows:\n                    save(scn, \"OutputSpatialValue\", pd.DataFrame(rows),\n                         [\"Iteration\", \"Timestep\", \"ServiceValueFile\", \"AssetValueFile\"])\n        except Exception as e:\n            warn(f\"SEEA-Sim: value maps not made: {e}\")\n\n    # Excel workbook\n    try:\n        import seea_outputs as so\n        fn = os.path.join(temp_folder(\"SEEASimWorkbook\"), f\"SEEA-Sim_accounts_scenario{scn.sid}.xlsx\")\n        so.write_workbook(fn, tables, {\"scenario\": f\"{scn.name} (scenario {scn.sid})\", \"version\": version(),\n                                       \"iterations\": state[\"Iteration\"].nunique(), \"currency\": currency,\n                                       \"services\": services(scn), \"periods\": periods})\n        save(scn, \"OutputWorkbook\", pd.DataFrame({\"File\": [fn]}), [\"File\"])\n        say(\"SEEA-Sim: Excel workbook saved (Results > SEEA-Sim Workbook)\")\n    except Exception as e:\n        warn(f\"SEEA-Sim: Excel workbook not made: {e}\")\n\n    say(\"SEEA-Sim: done\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except sa.AccountError as e:\n        say(f\"SEEA-Sim input error: {e}\", \"warning\")\n        raise\n    except Exception:\n        say(\"SEEA-Sim error:\\n\" + traceback.format_exc(), \"warning\")\n        raise\n"
}

In [3]:
# 2b. Bundle it into 4_seeasim/stsimseea-1-0-5.ssimpkg (files at the zip root + .metadata)
xml = PKG_FILES['package.xml']
ver = re.search(r'<package\s[^>]*version="([^"]+)"', xml).group(1)
ssim_min = re.search(r'minSyncroSimVersion="([^"]+)"', xml).group(1)
for old in glob.glob(os.path.join(OUT, 'stsimseea-*.ssimpkg')):
    os.remove(old)
PKG_PATH = os.path.join(OUT, f"stsimseea-{ver.replace('.', '-')}.ssimpkg")
with zipfile.ZipFile(PKG_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('.metadata', ('\ufeff<?xml version="1.0" encoding="utf-8"?>\r\n'
                             f'<package builtForSyncroSimVersion="{ssim_min}" />').encode('utf-8'))
    for name, text in PKG_FILES.items():
        z.writestr(name, text)
print(os.path.relpath(PKG_PATH, FOLDER), zipfile.ZipFile(PKG_PATH).namelist())

4_seeasim/stsimseea-1-0-5.ssimpkg ['.metadata', 'package.xml', 'seea_accounts.py', 'seea_outputs.py', 'stsimseea.yml', 'transformer.py']


In [4]:
# 3a. SEEA-Sim tables for JEM (examples/demak in the repo)
SEEA_FILES = {
 "add_seeasim.py": "\"\"\"\nAdd the SEEA-Sim example inputs to an ST-Sim library.\n\nCalled by build_and_test.py when a 4_seeasim folder sits next to it, or on\nits own for a library that already exists:\n\n    python add_seeasim.py --library JEM_cycle1.ssim --scenario \"Cycle1 2015 BaU\"\n    python add_seeasim.py ... --ssim \"C:\\\\Program Files\\\\SyncroSim Studio\"\n\nWhat it does, in order:\n  0. installs stsimseea from the .ssimpkg in this folder if SyncroSim does not\n     have that version yet\n  1. adds the stsimseea package to the library and turns on 'Use conda'\n     for it (SEEA-Sim's script runs in its own conda environment)\n  2. project: adds the SEEA stock and flow groups (only names not there yet)\n     and the Ecosystem Service list\n  3. scenario: adds the stock/flow group memberships (keeps existing rows),\n     turns on the ST-Sim output SEEA-Sim needs, saves every stsimseea_* table\n  4. sets the scenario pipeline: 1 ST-Sim, 2 SEEA-Sim Accounts\nWrites add_seeasim_log.txt next to this script.\n\"\"\"\nimport argparse\nimport glob\nimport os\n\nimport pandas as pd\nimport pysyncrosim as ps\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nP_DIR = os.path.join(HERE, \"project\")\nS_DIR = os.path.join(HERE, \"scenario\")\nLOG = []\n\n# Codes for SyncroSim List columns, tried if saving the labels fails.\nLIST_CODES = {\n    \"ServiceType\": {\"Provisioning\": 1, \"Regulating\": 2, \"Cultural\": 3},\n    \"ChangeType\": {\"Managed\": 1, \"Unmanaged\": 2, \"Catastrophic\": 3},\n    \"Role\": {\"Ecosystem pool\": 1, \"Atmosphere\": 2, \"Water\": 3, \"Long-term burial\": 4,\n             \"Other external\": 5},\n    \"Basis\": {\"Area\": 1, \"Stock group\": 2, \"Flow group\": 3},\n}\n\n\n_EXTERNAL_LOG = None\n\n\ndef log(level, msg):\n    if _EXTERNAL_LOG is not None:\n        _EXTERNAL_LOG(level, \"seeasim: \" + msg)\n        return\n    line = f\"[{level}] {msg}\"\n    print(line)\n    LOG.append(line)\n\n\ndef read_csv(path):\n    return pd.read_csv(path, dtype=str, keep_default_na=False).replace(\"\", None)\n\n\ndef norm(s):\n    return s.lower().replace(\"_\", \"\")\n\n\ndef match_columns(df, obj, name):\n    \"\"\"Rename CSV columns to the datasheet's own spelling (Id vs ID).\"\"\"\n    try:\n        real = list(obj.datasheets(name, empty=True).columns)\n    except Exception:\n        return df\n    if not real:\n        return df\n    lookup = {norm(c): c for c in real}\n    keep = {c: lookup[norm(c)] for c in df.columns if norm(c) in lookup}\n    dropped = [c for c in df.columns if c not in keep]\n    if dropped:\n        log(\"WARN\", f\"{name}: columns not in datasheet, dropped {dropped}; real columns {real}\")\n    return df[list(keep)].rename(columns=keep)\n\n\ndef save(obj, name, df, **kw):\n    df = match_columns(df, obj, name)\n    try:\n        obj.save_datasheet(name=name, data=df, **kw)\n        log(\"OK\", f\"{name}: {len(df)} rows\")\n        return\n    except Exception as e:\n        first = str(e).strip()[:300]\n    coded = df.copy()\n    changed = False\n    for col, codes in LIST_CODES.items():\n        if col in coded.columns:\n            coded[col] = coded[col].map(lambda v: codes.get(v, v) if v is not None else v)\n            changed = True\n    if changed:\n        try:\n            obj.save_datasheet(name=name, data=coded, **kw)\n            log(\"OK\", f\"{name}: {len(df)} rows (list values saved as codes)\")\n            return\n        except Exception as e:\n            first += \" | with codes: \" + str(e).strip()[:300]\n    log(\"FAIL\", f\"{name}: {first}\")\n\n\ndef add_new_names(proj, name, path):\n    \"\"\"Project definitions: append only names that are not there yet.\"\"\"\n    new = read_csv(path)\n    try:\n        cur = proj.datasheets(name)\n        have = set(cur[\"Name\"].astype(str)) if \"Name\" in cur.columns else set()\n    except Exception:\n        have = set()\n    new = new[~new[\"Name\"].isin(have)]\n    if new.empty:\n        log(\"OK\", f\"{name}: nothing new\")\n        return\n    save(proj, name, new, append=True)\n\n\ndef merge_membership(scn, name, path, key_cols):\n    new = match_columns(read_csv(path), scn, name)\n    try:\n        cur = scn.datasheets(name)\n    except Exception:\n        cur = pd.DataFrame(columns=new.columns)\n    both = pd.concat([cur[[c for c in new.columns if c in cur.columns]], new], ignore_index=True)\n    keys = [c for c in both.columns if norm(c) in {norm(k) for k in key_cols}]\n    both = both.drop_duplicates(subset=keys)\n    save(scn, name, both)\n\n\ndef require_output_options(scn, path):\n    name = \"stsim_OutputOptions\"\n    want = read_csv(path).iloc[0].to_dict()\n    try:\n        cur = scn.datasheets(name)\n    except Exception:\n        cur = pd.DataFrame()\n    row = cur.iloc[[0]].copy() if len(cur) else pd.DataFrame([{}])\n    for k, v in want.items():\n        row[k] = v\n    save(scn, name, row)\n\n\ndef set_pipeline(scn):\n    for first, second in ((\"stsim_Runtime\", \"stsimseea_Main\"), (\"ST-Sim\", \"SEEA-Sim Accounts\")):\n        try:\n            scn.save_datasheet(name=\"core_Pipeline\", data=pd.DataFrame(\n                {\"StageNameId\": [first, second], \"RunOrder\": [1, 2]}))\n            log(\"OK\", f\"pipeline: 1 {first}, 2 {second}\")\n            return\n        except Exception:\n            continue\n    log(\"WARN\", \"could not set the pipeline; set it in Studio: Scenario > Pipeline: \"\n                \"1 ST-Sim, 2 SEEA-Sim Accounts\")\n\n\ndef package_file_version():\n    \"\"\"(path, version) of the stsimseea-*.ssimpkg next to this script, or (None, None).\"\"\"\n    files = sorted(glob.glob(os.path.join(HERE, \"stsimseea-*.ssimpkg\")))\n    if not files:\n        return None, None\n    pkg = files[-1]\n    return pkg, os.path.basename(pkg)[len(\"stsimseea-\"):-len(\".ssimpkg\")].replace(\"-\", \".\")\n\n\ndef ensure_package(session):\n    \"\"\"Install stsimseea from the .ssimpkg next to this script if that version is not installed.\n    Returns the version the library should use (None = whatever is installed).\"\"\"\n    pkg, want = package_file_version()\n    if pkg is None:\n        log(\"WARN\", \"no stsimseea-*.ssimpkg in this folder; using the newest SEEA-Sim installed in SyncroSim\")\n        return None\n    try:\n        inst = session.packages()\n        have = set(inst.loc[inst[\"Name\"] == \"stsimseea\", \"Version\"].astype(str))\n        if want in have:\n            log(\"OK\", f\"SEEA-Sim {want} already installed (installed versions: {sorted(have)})\")\n            return want\n    except Exception as e:\n        log(\"WARN\", f\"could not list installed packages ({str(e)[:200]}); installing anyway\")\n    try:\n        session.install_packages(pkg)\n        log(\"OK\", f\"installed {os.path.basename(pkg)}\")\n    except Exception as e:\n        log(\"FAIL\", f\"could not install {os.path.basename(pkg)}: {str(e)[:300]}. \"\n                    \"Install it in Studio: File > Local Packages > Install from File\")\n    return want\n\n\ndef add_to_library(session, lib, proj, scn, external_log=None):\n    \"\"\"Everything in steps 0 to 4 on an open library, project and scenario.\"\"\"\n    global _EXTERNAL_LOG\n    _EXTERNAL_LOG = external_log\n    want = ensure_package(session)\n    try:\n        # Name the version: with several versions installed, pysyncrosim would\n        # otherwise take whichever SyncroSim lists last, which may be an older one.\n        if want:\n            lib.add_packages([\"stsimseea\"], versions=[want])  # lists: pysyncrosim 2.2 zips a version string into characters\n        else:\n            lib.add_packages(\"stsimseea\")\n    except Exception as e:\n        log(\"WARN\", f\"add_packages(stsimseea {want or ''}): {str(e)[:300]}\")\n    try:\n        used = lib.packages\n        used = set(used.loc[used[\"Name\"] == \"stsimseea\", \"Version\"].astype(str))\n    except Exception:\n        used = set()\n    if want and used and want not in used:\n        log(\"FAIL\", f\"library uses SEEA-Sim {sorted(used)}, not {want}. Uninstall the other version in \"\n                    \"Studio (File > Local Packages) and run this again\")\n    else:\n        log(\"OK\", f\"library uses SEEA-Sim {sorted(used) or want}\")\n    try:\n        # SEEA-Sim's transformer runs in its own conda environment; without this\n        # the stage ends at once with no output (\"'Use conda' property is disabled\").\n        lib.use_conda = True\n        log(\"OK\", \"library set to use conda (needed by SEEA-Sim)\")\n    except Exception as e:\n        log(\"WARN\", f\"could not turn on 'Use conda' ({str(e)[:200]}); turn it on in Studio: \"\n                    \"Library properties > Options > Use conda\")\n\n    add_new_names(proj, \"stsim_StockGroup\", os.path.join(P_DIR, \"stsim_StockGroup_additions.csv\"))\n    add_new_names(proj, \"stsim_FlowGroup\", os.path.join(P_DIR, \"stsim_FlowGroup_additions.csv\"))\n    add_new_names(proj, \"stsimseea_Service\", os.path.join(P_DIR, \"stsimseea_Service.csv\"))\n\n    merge_membership(scn, \"stsim_StockTypeGroupMembership\",\n                     os.path.join(S_DIR, \"stsim_StockTypeGroupMembership_additions.csv\"),\n                     [\"StockTypeId\", \"StockGroupId\"])\n    merge_membership(scn, \"stsim_FlowTypeGroupMembership\",\n                     os.path.join(S_DIR, \"stsim_FlowTypeGroupMembership_additions.csv\"),\n                     [\"FlowTypeId\", \"FlowGroupId\"])\n    require_output_options(scn, os.path.join(S_DIR, \"stsim_OutputOptions_required.csv\"))\n    for path in sorted(glob.glob(os.path.join(S_DIR, \"stsimseea_*.csv\"))):\n        save(scn, os.path.splitext(os.path.basename(path))[0], read_csv(path))\n    set_pipeline(scn)\n    _EXTERNAL_LOG = None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--library\", required=True)\n    ap.add_argument(\"--scenario\", required=True)\n    ap.add_argument(\"--project\", default=\"Definitions\")\n    ap.add_argument(\"--ssim\", default=None)\n    a = ap.parse_args()\n\n    session = ps.Session(location=a.ssim) if a.ssim else ps.Session()\n    lib = ps.library(name=a.library, session=session)\n    proj = lib.projects(name=a.project)\n    scn = proj.scenarios(name=a.scenario)\n    add_to_library(session, lib, proj, scn)\n\n    with open(os.path.join(HERE, \"add_seeasim_log.txt\"), \"w\", encoding=\"utf-8\") as f:\n        f.write(\"\\n\".join(LOG))\n    n_fail = sum(line.startswith(\"[FAIL\") for line in LOG)\n    print(f\"\\nDone: {n_fail} FAIL. Details in add_seeasim_log.txt\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "project/stsim_FlowGroup_additions.csv": "Name,Description,Units\nSEEA_all_carbon_flows,Every carbon flow type once (used by the SEEA-Sim carbon accounts),MgC\nSEEA_NPP,NPP flows from the atmosphere into biomass (GPP flow type),MgC\n",
 "project/stsim_StockGroup_additions.csv": "Name,Description,Units\nSEEA_AGB,SEEA-Sim pool: aboveground biomass (only AGB),MgC\nSEEA_BGB,SEEA-Sim pool: belowground biomass (only BGB),MgC\nSEEA_Soil,SEEA-Sim pool: soil organic carbon (only Soil),MgC\nSEEA_Atmosphere,SEEA-Sim external: atmosphere,MgC\nSEEA_Water,SEEA-Sim external: adjacent water,MgC\nSEEA_Buried,SEEA-Sim external: permanent burial,MgC\n",
 "project/stsimseea_Service.csv": "Name,ServiceType,Units,Description\nCapture Fisheries,Provisioning,kg,Wild fish catch supported by coastal water (150 kg/ha/yr)\nCrop Provisioning,Provisioning,kg,Rice and other crops (5000 kg/ha/yr)\nAquaculture Fishery,Provisioning,kg,Pond fish and shrimp (800 kg/ha/yr)\nFishery Nursery,Provisioning,ha,Mangrove area serving as nursery habitat\nCoastal Protection,Regulating,ha mangrove-equivalent,Wave and erosion protection; tidal flat counts as 1/3 of mangrove\nCarbon Sequestration,Regulating,MgC,Carbon taken up by vegetation: NPP flows from the atmosphere (flow group SEEA_NPP)\nEcotourism,Cultural,ha,Mangrove area used for ecotourism\nBiodiversity Habitat,Cultural,ha,Semi-natural vegetation habitat\n",
 "scenario/stsim_FlowTypeGroupMembership_additions.csv": "FlowTypeId,FlowGroupId,Value\nGPP,SEEA_all_carbon_flows,\nEmission,SEEA_all_carbon_flows,\nHumification,SEEA_all_carbon_flows,\nExport,SEEA_all_carbon_flows,\nSoil Export,SEEA_all_carbon_flows,\nBurial,SEEA_all_carbon_flows,\nSedimentation,SEEA_all_carbon_flows,\nGPP,SEEA_NPP,\n",
 "scenario/stsim_OutputOptions_required.csv": "SummaryOutputSC,SummaryOutputSCTimesteps,SummaryOutputTR,SummaryOutputTRTimesteps,SummaryOutputTRSC,SummaryOutputTRSCTimesteps\nYes,1,Yes,1,Yes,1\n",
 "scenario/stsim_StockTypeGroupMembership_additions.csv": "StockTypeId,StockGroupId,Value\nAGB,SEEA_AGB,\nBGB,SEEA_BGB,\nSoil,SEEA_Soil,\nAtmosphere,SEEA_Atmosphere,\nWater,SEEA_Water,\nBuried,SEEA_Buried,\n",
 "scenario/stsimseea_AccountingPeriod.csv": "OpeningTimestep,ClosingTimestep\n2015,2020\n2020,2022\n",
 "scenario/stsimseea_AssetValuation.csv": "StateClassId,DiscountRate,AssetLifeYears\n,0.03,100\nWater_body:All,0.03,100\nTidal_flat:All,0.03,100\nMangrove:All,0.03,100\nOther_vegetation:All,0.03,100\nSettlement:All,0.03,50\nCropland:All,0.03,50\nAquaculture:All,0.03,30\n",
 "scenario/stsimseea_CarbonFlowGroup.csv": "FlowGroupId\nSEEA_all_carbon_flows\n",
 "scenario/stsimseea_CarbonStockGroup.csv": "StockGroupId,Role\nSEEA_AGB,Ecosystem pool\nSEEA_BGB,Ecosystem pool\nSEEA_Soil,Ecosystem pool\nSEEA_Atmosphere,Atmosphere\nSEEA_Water,Water\nSEEA_Buried,Long-term burial\n",
 "scenario/stsimseea_Options.csv": "ReportByStratum,Currency,MapOutput\nNo,IDR (2022 prices),Yes\n",
 "scenario/stsimseea_ServicePrice.csv": "ServiceId,Timestep,Price\nCapture Fisheries,,20000\nCrop Provisioning,,6000\nAquaculture Fishery,,56250\nFishery Nursery,,8000000\nCoastal Protection,,15000000\nCarbon Sequestration,,238333\nEcotourism,,12500000\nBiodiversity Habitat,,10000000\n",
 "scenario/stsimseea_ServiceSupply.csv": "ServiceId,StratumId,StateClassId,Basis,StockGroupId,FlowGroupId,Coefficient\nCapture Fisheries,,Water_body:All,Area,,,150\nCrop Provisioning,,Cropland:All,Area,,,5000\nAquaculture Fishery,,Aquaculture:All,Area,,,800\nFishery Nursery,,Mangrove:All,Area,,,1\nCoastal Protection,,Mangrove:All,Area,,,1\nCoastal Protection,,Tidal_flat:All,Area,,,0.3333\nCarbon Sequestration,,,Flow group,,SEEA_NPP,1\nEcotourism,,Mangrove:All,Area,,,1\nBiodiversity Habitat,,Other_vegetation:All,Area,,,1\n",
 "scenario/stsimseea_TransitionTypeClass.csv": "TransitionTypeId,ChangeType\nUrbanization__Mangrove_to_Settlement,Managed\nAgriculture_expansion_Mangrove__Mangrove_to_Cropland,Managed\nAquaculture_expansion__Mangrove_to_Aquaculture,Managed\nUrbanization__Other_vegetation_to_Settlement,Managed\nAgriculture_expansion_OV__Other_vegetation_to_Cropland,Managed\nAquaculture_expansion__Other_vegetation_to_Aquaculture,Managed\nUrbanization__Cropland_to_Settlement,Managed\nAquaculture_expansion__Cropland_to_Aquaculture,Managed\nUrbanization__Aquaculture_to_Settlement,Managed\nAgriculture_expansion_Aqua__Aquaculture_to_Cropland,Managed\nSedimentation__Water_body_to_Tidal_flat,Unmanaged\nMangrove_recruitment__Water_body_to_Mangrove,Unmanaged\nTidal_flat_to_Waterbody__Tidal_flat_to_Water_body,Unmanaged\nMangrove_recruitment__Tidal_flat_to_Mangrove,Unmanaged\nInundation__Mangrove_to_Water_body,Unmanaged\nInundation__Mangrove_to_Tidal_flat,Unmanaged\nMangrove_recruitment__Cropland_to_Mangrove,Unmanaged\nColonization__Cropland_to_Other_vegetation,Unmanaged\nMangrove_recruitment__Aquaculture_to_Mangrove,Unmanaged\nColonization__Aquaculture_to_Other_vegetation,Unmanaged\n",
 "README.md": "# Example: Demak mangrove coast (validation 2015 to 2022)\n\nSEEA-Sim inputs for the JEM ST-Sim model of the Demak coast (7 classes, 20 transition types, carbon stocks AGB, BGB and Soil). Names match that library; for another model, change the names and values.\n\n## Files\n\nAll files are in `demak/`.\n\n| File | Level | What it sets |\n|---|---|---|\n| `demak/project/stsimseea_Service.csv` | Project | 8 ecosystem services with type and physical unit |\n| `demak/project/stsim_StockGroup_additions.csv` | Project (ST-Sim) | One stock group per carbon stock type (`SEEA_AGB` ... `SEEA_Buried`). The model's own groups overlap (Total and Biomass both hold AGB), and the carbon accounts need groups that don't |\n| `demak/project/stsim_FlowGroup_additions.csv` | Project (ST-Sim) | `SEEA_all_carbon_flows` (every flow type once) and `SEEA_NPP` (the GPP flow type, for Carbon Sequestration) |\n| `demak/scenario/stsim_StockTypeGroupMembership_additions.csv` | Scenario (ST-Sim) | Puts each stock type in its SEEA group |\n| `demak/scenario/stsim_FlowTypeGroupMembership_additions.csv` | Scenario (ST-Sim) | Puts the flow types in the two SEEA flow groups |\n| `demak/scenario/stsim_OutputOptions_required.csv` | Scenario (ST-Sim) | Turns on transitions-by-state-class output every timestep |\n| `demak/scenario/stsimseea_Options.csv` | Scenario | Whole landscape (no stratum split), currency label, value maps on |\n| `demak/scenario/stsimseea_AccountingPeriod.csv` | Scenario | 2015 to 2020 and 2020 to 2022 |\n| `demak/scenario/stsimseea_TransitionTypeClass.csv` | Scenario | 10 human conversions Managed, 10 natural processes Unmanaged |\n| `demak/scenario/stsimseea_CarbonStockGroup.csv` | Scenario | AGB, BGB, Soil = ecosystem pools; Atmosphere, Water, Buried = external |\n| `demak/scenario/stsimseea_CarbonFlowGroup.csv` | Scenario | Use `SEEA_all_carbon_flows` |\n| `demak/scenario/stsimseea_ServiceSupply.csv` | Scenario | How each service's quantity is computed |\n| `demak/scenario/stsimseea_ServicePrice.csv` | Scenario | IDR per physical unit |\n| `demak/scenario/stsimseea_AssetValuation.csv` | Scenario | 3% discount rate; asset life 100 years (natural), 50 (settlement, cropland), 30 (ponds); same as `06_AssetValuationParams.csv` |\n| `demak/add_seeasim.py` | | Loads all of the above into a library (called by `build_and_test.py`) |\n| `demak/build_jem_seeasim_library.ipynb` | | Colab: puts this folder, the package file and the `build_and_test.py` hook into the JEM folder on Drive |\n\n## Use with the JEM inputs\n\n`build_jem_seeasim_library.ipynb` (Colab) writes the `demak/` files into the JEM inputs folder on Drive as `4_seeasim/`, adds the `stsimseea-1-0-5.ssimpkg` package file to it, and adds a few lines to `build_and_test.py` so that it loads `4_seeasim/` when the folder is there. It also checks every name against that folder's ST-Sim tables. It does not need SyncroSim.\n\nThen, on the laptop, in the JEM folder:\n\n```\npip install pysyncrosim pandas\npython build_and_test.py --full --no-run --ssim \"C:\\Program Files\\SyncroSim Studio\"\n```\n\n`build_and_test.py` builds the ST-Sim library as before, then calls `4_seeasim/add_seeasim.py`, which:\n- installs SEEA-Sim from the `.ssimpkg` if SyncroSim does not have that version yet;\n- adds it to the library and turns on 'Use conda' for the library (SEEA-Sim's script runs in its own conda environment);\n- adds the SEEA stock and flow groups and the service list;\n- saves the scenario tables and turns on the ST-Sim output SEEA-Sim needs;\n- sets the pipeline (1 ST-Sim, 2 SEEA-Sim Accounts).\n\nIts lines are in `build_log.txt`, marked `seeasim:`. Running it twice is safe.\n\nFor a library that already exists: `python add_seeasim.py --library JEM_cycle1.ssim --scenario \"Cycle1 2015 BaU\" --ssim \"C:\\Program Files\\SyncroSim Studio\"`.\n\n## Where the values come from, and what to check\n\nPhysical quantities and per-hectare values are the earlier SEEA-Sim drafts' values (`3_seeasim_addon` in the JEM inputs). Prices are those values divided by the quantity per hectare:\n\n| Service | Basis | Quantity | Price (IDR) | Same as |\n|---|---|---|---|---|\n| Capture Fisheries | Water body area | 150 kg/ha/yr | 20,000 /kg | 3.0 M/ha |\n| Crop Provisioning | Cropland area | 5,000 kg/ha/yr | 6,000 /kg | 30 M/ha |\n| Aquaculture Fishery | Pond area | 800 kg/ha/yr | 56,250 /kg | 45 M/ha |\n| Fishery Nursery | Mangrove area | 1 ha | 8 M /ha | 8 M/ha |\n| Coastal Protection | Mangrove area x 1, tidal flat x 1/3 | ha mangrove-equivalent | 15 M /ha | 15 M/ha mangrove, 5 M/ha tidal flat |\n| Carbon Sequestration | NPP flows (flow group `SEEA_NPP`) | MgC | 238,333 /MgC | 238,333 /MgC |\n| Ecotourism | Mangrove area | 1 ha | 12.5 M /ha | 12.5 M/ha |\n| Biodiversity Habitat | Other vegetation area | 1 ha | 10 M /ha | 10 M/ha |\n\nService names, units, the carbon price and the asset-valuation table (`DiscountRate`, `AssetLifeYears`, one row per class) follow `3_seeasim_addon` in the JEM inputs. `PriceGrowthRate` is not used: give dated rows in Service Prices instead.\n\nChanges from the earlier draft:\n- **Carbon sequestration on Other vegetation (350 MgC/ha)** is replaced by the model's own NPP flows for every class, so the unit question goes away.\n- **Carbon storage** (pricing the AGB, BGB and soil stock every year) is left out of the services: it would value the whole stock again each year. The stock is reported in the carbon stock account instead.\n- **Carbon Sequestration** here is gross uptake (NPP). Net sequestration would subtract soil respiration; decide which one the paper reports.\n\nBefore using these numbers in results:\n- The prices are placeholders from the drafts. Replace them with sourced Demak values and a stated price year.\n- Inundation is classed Unmanaged. If you treat abrasion as a disaster, change it to Catastrophic and it moves to \"Catastrophic losses\" in the asset account.\n- The validation scenario runs 2015 to 2022. For the projections, use 2022 to 2026 and 2026 to 2031 as periods.\n"
}

In [5]:
# 3b. Write them to 4_seeasim/
for rel, text in SEEA_FILES.items():
    path = os.path.join(OUT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8', newline='') as f:
        f.write(text)
    print('wrote', os.path.relpath(path, FOLDER))

wrote 4_seeasim/add_seeasim.py
wrote 4_seeasim/project/stsim_FlowGroup_additions.csv
wrote 4_seeasim/project/stsim_StockGroup_additions.csv
wrote 4_seeasim/project/stsimseea_Service.csv
wrote 4_seeasim/scenario/stsim_FlowTypeGroupMembership_additions.csv
wrote 4_seeasim/scenario/stsim_OutputOptions_required.csv
wrote 4_seeasim/scenario/stsim_StockTypeGroupMembership_additions.csv
wrote 4_seeasim/scenario/stsimseea_AccountingPeriod.csv
wrote 4_seeasim/scenario/stsimseea_AssetValuation.csv
wrote 4_seeasim/scenario/stsimseea_CarbonFlowGroup.csv
wrote 4_seeasim/scenario/stsimseea_CarbonStockGroup.csv
wrote 4_seeasim/scenario/stsimseea_Options.csv
wrote 4_seeasim/scenario/stsimseea_ServicePrice.csv
wrote 4_seeasim/scenario/stsimseea_ServiceSupply.csv
wrote 4_seeasim/scenario/stsimseea_TransitionTypeClass.csv
wrote 4_seeasim/README.md


In [6]:
# 4. Check every name against this folder's ST-Sim tables
P = os.path.join(FOLDER, '1_project_definitions')
names = lambda f: set(pd.read_csv(os.path.join(P, f), dtype=str)['Name'].str.strip())
classes, ttypes = names('05_stsim_StateClass.csv'), names('06_stsim_TransitionType.csv')
stypes, ftypes = names('13_stsim_StockType.csv'), names('16_stsim_FlowType.csv')
rd = lambda f: pd.read_csv(os.path.join(OUT, f), dtype=str, keep_default_na=False)
problems = []
def check(values, allowed, what):
    problems.extend(f'{what}: {v!r} is not in the model' for v in sorted(set(values) - {''} - allowed))
ttc = rd('scenario/stsimseea_TransitionTypeClass.csv')
check(ttc.TransitionTypeId, ttypes, 'Transition type')
problems += [f'Transition type {t!r} has no class (counts as Unmanaged)' for t in sorted(ttypes - set(ttc.TransitionTypeId))]
check(rd('scenario/stsimseea_ServiceSupply.csv').StateClassId, classes, 'Supply rule class')
check(rd('scenario/stsimseea_AssetValuation.csv').StateClassId, classes, 'Asset valuation class')
check(rd('scenario/stsim_StockTypeGroupMembership_additions.csv').StockTypeId, stypes, 'Stock type')
fm = rd('scenario/stsim_FlowTypeGroupMembership_additions.csv')
check(fm.FlowTypeId, ftypes, 'Flow type')
problems += [f'Flow type {t!r} missing from SEEA_all_carbon_flows' for t in sorted(ftypes - set(fm[fm.FlowGroupId == 'SEEA_all_carbon_flows'].FlowTypeId))]
services = set(rd('project/stsimseea_Service.csv').Name)
check(rd('scenario/stsimseea_ServiceSupply.csv').ServiceId, services, 'Supply rule service')
check(rd('scenario/stsimseea_ServicePrice.csv').ServiceId, services, 'Price service')
groups = set(pd.read_csv(os.path.join(P, '14_stsim_StockGroup.csv'), dtype=str)['Name'])
clash = groups & set(rd('project/stsim_StockGroup_additions.csv').Name)
if clash:
    print('Note: stock groups already in 14_stsim_StockGroup.csv (fine, they are skipped):', clash)
print('\n'.join(problems) if problems else f'All names match: {len(classes)} classes, {len(ttypes)} transition types, '
      f'{len(stypes)} stock types, {len(ftypes)} flow types, {len(services)} services.')

All names match: 7 classes, 20 transition types, 6 stock types, 7 flow types, 8 services.


In [7]:
# 5. Hook 4_seeasim into build_and_test.py (keeps the original once as _OLD_build_and_test_before_seeasim.py)
HOOK = "    seea_dir = os.path.join(HERE, \"4_seeasim\")\n    if os.path.isdir(seea_dir):\n        import sys\n        sys.path.insert(0, seea_dir)\n        import add_seeasim\n        add_seeasim.add_to_library(session, lib, proj, scn, external_log=log)\n"
DOC = "\nIf a 4_seeasim folder sits next to this script, the library also gets SEEA-Sim:\nthe stsimseea package (installed from 4_seeasim/stsimseea-*.ssimpkg if needed),\nits tables, the SEEA stock and flow groups, the extra ST-Sim output and the\npipeline 1 ST-Sim, 2 SEEA-Sim Accounts (see 4_seeasim/README.md).\n"
bt = os.path.join(FOLDER, 'build_and_test.py')
src = open(bt, encoding='utf-8').read()
if 'add_seeasim.add_to_library' in src:
    print('build_and_test.py already loads 4_seeasim; nothing to change')
else:
    backup = os.path.join(FOLDER, '_OLD_build_and_test_before_seeasim.py')
    if not os.path.exists(backup):
        shutil.copy2(bt, backup)
    anchor = '    set_pipeline(scn)\n'
    assert src.count(anchor) == 1, 'build_and_test.py has changed: add the hook by hand after set_pipeline(scn)'
    src = src.replace(anchor, anchor + HOOK)
    end = src.index('"""', 3)
    src = src[:end] + DOC + src[end:]
    with open(bt, 'w', encoding='utf-8') as f:
        f.write(src)
    print('build_and_test.py now loads 4_seeasim; original kept as', os.path.basename(backup))
i = src.index(anchor if 'anchor' in dir() else '    set_pipeline(scn)')
print(src[i:i + 400])

build_and_test.py already loads 4_seeasim; nothing to change
    set_pipeline(scn)
    seea_dir = os.path.join(HERE, "4_seeasim")
    if os.path.isdir(seea_dir):
        import sys
        sys.path.insert(0, seea_dir)
        import add_seeasim
        add_seeasim.add_to_library(session, lib, proj, scn, external_log=log)
    try:
        p = scn.datasheets("stsim_InitialConditionsSpatialProperties")
        log("OK", "raster grid: " + ", ".join(f"{c}={p[c].


In [8]:
# 6. What is in 4_seeasim now, and what to run on the laptop
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        print(os.path.relpath(os.path.join(root, f), FOLDER))
print('''
On the laptop, in this folder:
    pip install pysyncrosim pandas
    python build_and_test.py --full --no-run --ssim "C:\\Program Files\\SyncroSim Studio"
Then check the seeasim: lines in build_log.txt and open JEM_cycle1.ssim in Studio.''')

4_seeasim/README.md
4_seeasim/README_seeasim.md
4_seeasim/add_seeasim.py
4_seeasim/stsimseea-1-0-5.ssimpkg
4_seeasim/project/stsim_FlowGroup_additions.csv
4_seeasim/project/stsim_StockGroup_additions.csv
4_seeasim/project/stsimseea_Service.csv
4_seeasim/scenario/stsim_FlowTypeGroupMembership_additions.csv
4_seeasim/scenario/stsim_OutputOptions_required.csv
4_seeasim/scenario/stsim_StockTypeGroupMembership_additions.csv
4_seeasim/scenario/stsimseea_AccountingPeriod.csv
4_seeasim/scenario/stsimseea_AssetValuation.csv
4_seeasim/scenario/stsimseea_CarbonFlowGroup.csv
4_seeasim/scenario/stsimseea_CarbonStockGroup.csv
4_seeasim/scenario/stsimseea_Options.csv
4_seeasim/scenario/stsimseea_ServicePrice.csv
4_seeasim/scenario/stsimseea_ServiceSupply.csv
4_seeasim/scenario/stsimseea_TransitionTypeClass.csv

On the laptop, in this folder:
    pip install pysyncrosim pandas
    python build_and_test.py --full --no-run --ssim "C:\Program Files\SyncroSim Studio"
Then check the seeasim: lines in build